# IDH Mutation Prediction in Glioma — Full Pipeline
### Beyond Histological Grading: Radiomics, CNN, and Hybrid Fusion

**Datasets:** UCSF-PDGM (training) · EGD (external validation) · TCGA BraTS 2021 (external validation)

**Tracks:** Track A — 7 sequences (T1, T1c, T2, FLAIR, ADC, FA, MD) · Track B — 4 sequences (T1, T1c, T2, FLAIR)

**Arms:** Radiomics (PyRadiomics + SVM/XGBoost) · CNN (ResNet18) · Hybrid Fusion

---
**Cell index:**
- Cell 1: Setup — install dependencies, verify datasets mounted
- Cell 2: Labels — build patient list, binarize IDH
- Cell 3: Splits — stratified 5-fold, unit test
- Cell 4: Preprocessing — clip, normalize, build CNN tensors + summary
- Cell 5: CNN training — Track A (ResNet18 7-channel)
- Cell 6: CNN training — Track B (ResNet18 4-channel)
- Cell 7: Radiomics feature extraction
- Cell 8: Radiomics feature selection + classification (SVM/XGBoost) + SHAP
- Cell 9: Hybrid fusion — embedding extraction + PCA + concatenation
- Cell 10: Fusion classifier (SVM/XGBoost) — early and late fusion
- Cell 11: External validation — EGD + TCGA (Track B models only)
- Cell 12: Ablation studies — 4-seq vs 5-seq vs 7-seq
- Cell 13: Statistical tests — DeLong, bootstrap CI, Bonferroni
- Cell 14: Figures — ROC curves, SHAP, Grad-CAM, confusion matrices
- Cell 15: Results table + final summary

## Cell 1 — Setup
**Run every new session before anything else.**

In [1]:
!pip install nibabel SimpleITK scikit-image -q

import os, json, glob
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
from skimage.transform import resize
from sklearn.model_selection import StratifiedKFold

WORKING = '/kaggle/working'
os.makedirs(f'{WORKING}/trackA_slices', exist_ok=True)
# Track B derived at runtime from trackA[:, :4, :, :] — no separate directory needed

# Input paths — all 10 batch datasets
KAGGLE_USER  = 'adesaladaniel'
BATCH_ROOTS  = [f'/kaggle/input/datasets/{KAGGLE_USER}/ucsf-pdgm-batch-{i:02d}' for i in range(1, 11)]

# Sequence filenames
TRACK_A_SEQS = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TRACK_B_SEQS = ['T1', 'T1c', 'T2', 'FLAIR']
MASK_SEQS    = ['tumor_segmentation', 'brain_segmentation']

print('Dependencies loaded.')
print(f'Batch roots found: {sum(os.path.exists(b) for b in BATCH_ROOTS)}/10')
for b in BATCH_ROOTS:
    status = 'OK     ' if os.path.exists(b) else 'MISSING'
    print(f'  {status}  {b}')

Dependencies loaded.
Batch roots found: 10/10
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-02
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-03
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-04
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-05
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-06
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-07
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-08
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-09
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10


## Cell 2 — Labels
**build patient list, binarize IDH**

In [2]:
import requests

# ── DISCOVER ALL PATIENT FOLDERS ─────────────────────────────────────────────
patient_dirs = {}
for batch_root in BATCH_ROOTS:
    if not os.path.exists(batch_root):
        continue
    for entry in sorted(os.listdir(batch_root)):
        if entry.startswith('UCSF-PDGM-'):
            full_path = os.path.join(batch_root, entry)
            if os.path.isdir(full_path):
                patient_dirs[entry] = full_path

print(f'Patient folders found: {len(patient_dirs)}')
if len(patient_dirs) == 0:
    print('ERROR: No patient folders found.')
    raise SystemExit

# Print first 3 and last 3 to verify structure
sample = sorted(patient_dirs.keys())
for pid in sample[:3] + sample[-3:]:
    print(f'  {pid}  →  {patient_dirs[pid]}')

print()

# ── LOAD METADATA AND BINARIZE IDH ───────────────────────────────────────────
meta_path = f'{WORKING}/meta.csv'
if not os.path.exists(meta_path):
    print('Downloading metadata CSV...')
    r = requests.get('https://www.cancerimagingarchive.net/wp-content/uploads/UCSF-PDGM-metadata.csv')
    with open(meta_path, 'wb') as f:
        f.write(r.content)
    print('Downloaded.')

df_meta = pd.read_csv(meta_path)

# Standardize patient ID to UCSF-PDGM-XXXX
df_meta['ID_padded'] = df_meta['ID'].apply(
    lambda p: f'UCSF-PDGM-{int(str(p).split("-")[-1]):04d}'
)

# Binarize IDH: wildtype=0, all mutation variants=1
def binarize_idh(val):
    if pd.isna(val):
        return None
    v = str(val).strip().lower()
    if v == 'wildtype':
        return 0
    return 1

df_meta['IDH_binary'] = df_meta['IDH'].apply(binarize_idh)

# Exclude follow-up scans
FOLLOWUP_IDS = [
    'UCSF-PDGM-0433', 'UCSF-PDGM-0431', 'UCSF-PDGM-0396',
    'UCSF-PDGM-0429', 'UCSF-PDGM-0409', 'UCSF-PDGM-0391'
]
df_labels = df_meta[
    df_meta['IDH_binary'].notna() &
    ~df_meta['ID_padded'].isin(FOLLOWUP_IDS)
][['ID_padded', 'IDH', 'IDH_binary', 'WHO CNS Grade',
   'Final pathologic diagnosis (WHO 2021)', 'MGMT status']].copy()
df_labels = df_labels.reset_index(drop=True)
df_labels.rename(columns={'ID_padded': 'patient_id'}, inplace=True)

# Keep only patients we have folders for
df_labels = df_labels[df_labels['patient_id'].isin(patient_dirs)].reset_index(drop=True)
df_labels['folder_path'] = df_labels['patient_id'].map(patient_dirs)

wt = (df_labels['IDH_binary'] == 0).sum()
mt = (df_labels['IDH_binary'] == 1).sum()
print(f'Patients with labels + folders : {len(df_labels)}  (expect 495)')
print(f'IDH Wildtype                   : {wt}')
print(f'IDH Mutant                     : {mt}')
print(f'Class ratio                    : {wt/mt:.2f}:1')
print()
print('Grade distribution:')
print(df_labels['WHO CNS Grade'].value_counts().sort_index().to_string())

df_labels.to_csv(f'{WORKING}/labels.csv', index=False)
print(f'\nSaved: {WORKING}/labels.csv')

Patient folders found: 495
  UCSF-PDGM-0004  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0004
  UCSF-PDGM-0005  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0005
  UCSF-PDGM-0007  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0007
  UCSF-PDGM-0539  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0539
  UCSF-PDGM-0540  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0540
  UCSF-PDGM-0541  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0541

Downloaded.
Patients with labels + folders : 495  (expect 495)
IDH Wildtype                   : 392
IDH Mutant                     : 103
Class ratio                    : 3.81:1

Grade distribution:
WHO CNS Grade
2     56
3     43
4    396

Saved: /kaggle/working/labels.csv


## Cell 3: Splits 
**stratified 5-fold, unit test**

In [3]:
# ── STRATIFIED 5-FOLD SPLIT ───────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

patient_ids = df_labels['patient_id'].values
idh_labels  = df_labels['IDH_binary'].values.astype(int)
fold_assignments = np.zeros(len(df_labels), dtype=int)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(patient_ids, idh_labels)):
    fold_assignments[val_idx] = fold_idx

df_labels['fold'] = fold_assignments
df_labels.to_csv(f'{WORKING}/labels.csv', index=False)

# Save fold assignments as JSON
splits = {}
for fold in range(5):
    fold_patients = df_labels[df_labels['fold'] == fold]['patient_id'].tolist()
    splits[f'fold_{fold}'] = fold_patients

with open(f'{WORKING}/patient_splits.json', 'w') as f:
    json.dump(splits, f, indent=2)

# ── UNIT TEST: ZERO PATIENT OVERLAP ──────────────────────────────────────────
print('Running data leakage unit test...')
print()
all_passed = True
for fold in range(5):
    val_patients   = set(splits[f'fold_{fold}'])
    train_patients = set()
    for other_fold in range(5):
        if other_fold != fold:
            train_patients.update(splits[f'fold_{other_fold}'])
    overlap = val_patients & train_patients
    if overlap:
        print(f'  FAIL fold {fold}: {len(overlap)} patients in both train and val!')
        all_passed = False
    else:
        val_wt = sum(1 for p in val_patients
                     if df_labels.loc[df_labels['patient_id']==p, 'IDH_binary'].values[0] == 0)
        val_mt = len(val_patients) - val_wt
        print(f'  PASS fold {fold}: {len(val_patients)} patients — {val_wt} wildtype / {val_mt} mutant')

print()
if all_passed:
    print('All folds pass — zero patient ID overlap confirmed.')
    print(f'Saved: {WORKING}/patient_splits.json')
else:
    print('FAILED — do not proceed.')
    raise SystemExit

Running data leakage unit test...

  PASS fold 0: 99 patients — 79 wildtype / 20 mutant
  PASS fold 1: 99 patients — 79 wildtype / 20 mutant
  PASS fold 2: 99 patients — 78 wildtype / 21 mutant
  PASS fold 3: 99 patients — 78 wildtype / 21 mutant
  PASS fold 4: 99 patients — 78 wildtype / 21 mutant

All folds pass — zero patient ID overlap confirmed.
Saved: /kaggle/working/patient_splits.json


## Cell 4: Preprocessing 
**clip, normalize, build CNN tensors**

In [ ]:
if False:
    # ── OLD CELL 4 — PREPROCESSING (RETIRED 2026-06-26) ──────────────────────────
    # This cell has been retired. Tensors are now permanently stored at:
    #   adesaladaniel/glioma-idh-tensors
    # Mounted at: /kaggle/input/glioma-idh-tensors/
    # Use new Cell 4 (verification cell) instead.
    # This code is kept for reference only — do NOT run it.
    # ─────────────────────────────────────────────────────────────────────────────
    
    # ── CONSTANTS ─────────────────────────────────────────────────────────────────
    TRACK_A_SEQS  = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
    TUMOR_SEQ     = 'tumor_segmentation'
    BRAIN_SEQ     = 'brain_segmentation'
    SLICE_SIZE    = 224
    TUMOR_THR     = 0.001
    
    # ── RELOAD STATE ──────────────────────────────────────────────────────────────
    df_labels     = pd.read_csv(f'{WORKING}/labels.csv')
    progress_path = f'{WORKING}/progress.json'
    
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            progress = json.load(f)
        completed_raw = set(progress.get('completed', []))
        completed = set(
            p for p in completed_raw
            if os.path.exists(f'{WORKING}/trackA_slices/{p}.npy')
        )
        lost = completed_raw - completed
        if lost:
            print(f'WARNING: {len(lost)} patients marked done but .npy files missing — will reprocess')
        failed     = progress.get('failed', {})
        # Only keep slice_meta for patients confirmed on disk
        slice_meta = [s for s in progress.get('slice_meta', [])
                      if s['patient_id'] in completed]
        print(f'RESUMING — {len(completed)} confirmed on disk, {len(failed)} failed')
    else:
        completed  = set()
        failed     = {}
        slice_meta = []
        print('Starting fresh')
    
    # ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────
    def get_nii_path(patient_dir, patient_id, seq_name):
        """
        Handles nested folder structure:
        UCSF-PDGM-0004/
          UCSF-PDGM-0004_T1.nii/
            UCSF-PDGM-0004_T1.nii  <-- actual file
        Also handles flat structure as fallback.
        """
        filename = f'{patient_id}_{seq_name}.nii'
        nested   = os.path.join(patient_dir, filename, filename)
        flat     = os.path.join(patient_dir, filename)
        if os.path.isfile(nested):
            return nested
        elif os.path.isfile(flat):
            return flat
        return None
    
    def load_volume(path):
        img = nib.load(path)
        return img.get_fdata(dtype=np.float32)
    
    def clip_and_normalize(vol, brain_mask):
        brain_voxels = vol[brain_mask > 0]
        if brain_voxels.size == 0:
            return vol
        # Clip to 1st-99th percentile within brain mask
        p1  = np.percentile(brain_voxels, 1)
        p99 = np.percentile(brain_voxels, 99)
        vol = np.clip(vol, p1, p99)
        # Z-score normalize within brain mask
        brain_voxels = vol[brain_mask > 0]
        mean = brain_voxels.mean()
        std  = brain_voxels.std()
        if std < 1e-8:
            vol = np.zeros_like(vol)
        else:
            vol = (vol - mean) / std
        vol[brain_mask == 0] = 0.0
        return vol
    
    def select_tumor_slices(tumor_mask):
        n_slices   = tumor_mask.shape[2]
        slice_area = tumor_mask.shape[0] * tumor_mask.shape[1]
        selected   = []
        for s in range(n_slices):
            coverage = (tumor_mask[:, :, s] > 0).sum() / slice_area
            if coverage >= TUMOR_THR:
                selected.append(s)
        return selected
    
    def resize_slice(slc):
        return resize(slc, (SLICE_SIZE, SLICE_SIZE),
                      order=1, mode='constant',
                      anti_aliasing=True,
                      preserve_range=True).astype(np.float32)
    
    def preprocess_patient(patient_id, patient_dir):
        try:
            # Resolve file paths
            tumor_path = get_nii_path(patient_dir, patient_id, TUMOR_SEQ)
            brain_path = get_nii_path(patient_dir, patient_id, BRAIN_SEQ)
    
            if tumor_path is None:
                return None, 0, f'Cannot find tumor_segmentation.nii'
            if brain_path is None:
                return None, 0, f'Cannot find brain_segmentation.nii'
    
            # Load and binarize masks
            tumor_mask = load_volume(tumor_path)
            brain_mask = load_volume(brain_path)
            tumor_mask = (tumor_mask > 0).astype(np.float32)
            brain_mask = (brain_mask > 0).astype(np.float32)
    
            # Select tumor-containing axial slices
            selected = select_tumor_slices(tumor_mask)
            if len(selected) == 0:
                return None, 0, 'No tumor slices above 0.1% coverage threshold'
    
            # Load, normalize, extract slices for each sequence
            channel_arrays = []
            for seq in TRACK_A_SEQS:
                seq_path = get_nii_path(patient_dir, patient_id, seq)
                if seq_path is None:
                    return None, 0, f'Cannot find {seq}.nii'
                vol = load_volume(seq_path)
                vol = clip_and_normalize(vol, brain_mask)
                seq_slices = np.stack(
                    [resize_slice(vol[:, :, s]) for s in selected],
                    axis=0
                )  # shape: (n_slices, 224, 224)
                channel_arrays.append(seq_slices)
    
            # Stack into Track A tensor: (n_slices, 7, 224, 224)
            # Track B = trackA[:, :4, :, :] — derived at runtime, not saved separately
            trackA = np.stack(channel_arrays, axis=1).astype(np.float16)
    
            return trackA, len(selected), None
    
        except Exception as e:
            return None, 0, str(e)
    
    # ── MAIN LOOP ─────────────────────────────────────────────────────────────────
    remaining = [p for p in df_labels['patient_id'].tolist() if p not in completed]
    
    print(f'Patients remaining : {len(remaining)}')
    print(f'Saving to          : {WORKING}/trackA_slices/')
    print(f'Track B derived at runtime from first 4 channels of Track A')
    print('='*60)
    
    for i, patient_id in enumerate(remaining):
        row         = df_labels[df_labels['patient_id'] == patient_id].iloc[0]
        patient_dir = row['folder_path']
        fold        = int(row['fold'])
        idh_label   = int(row['IDH_binary'])
    
        trackA, n_slices, error = preprocess_patient(patient_id, patient_dir)
    
        if error:
            failed[patient_id] = error
            print(f'  [{i+1}/{len(remaining)}] FAIL {patient_id} — {error}')
        else:
            np.save(f'{WORKING}/trackA_slices/{patient_id}.npy', trackA)
            for s in range(n_slices):
                slice_meta.append({
                    'patient_id': patient_id,
                    'slice_idx':  s,
                    'fold':       fold,
                    'idh_label':  idh_label
                })
            completed.add(patient_id)
            # Remove from failed if it was previously failed
            failed.pop(patient_id, None)
            print(f'  [{i+1}/{len(remaining)}] OK  {patient_id} — {n_slices} slices | fold {fold} | IDH {idh_label}')
    
        # Save progress after every patient
        with open(progress_path, 'w') as f:
            json.dump({
                'completed':  list(completed),
                'failed':     failed,
                'slice_meta': slice_meta
            }, f)
    
    print()
    print('='*60)
    print(f'Completed : {len(completed)}/{len(df_labels)}')
    print(f'Failed    : {len(failed)}')
    if failed:
        print('Failed patients:')
        for pid, reason in failed.items():
            print(f'  {pid}: {reason}')
    if len(completed) == len(df_labels):
        print('All patients done — proceed to Cell 5.')
    else:
        print('Re-run Cell 1 then Cell 4 to continue.')
    
    
    
    # ── SUMMARY ───────────────────────────────────────────────────────────────────
    df_slices = pd.DataFrame(slice_meta)
    df_slices.to_csv(f'{WORKING}/slice_metadata.csv', index=False)
    
    print()
    print('='*60)
    print('PREPROCESSING SUMMARY')
    print('='*60)
    print(f'Patients processed     : {len(completed)}')
    print(f'Patients failed        : {len(failed)}')
    print(f'Total slices           : {len(df_slices)}')
    print(f'Average slices/patient : {len(df_slices)/len(completed):.1f}')
    print()
    print('Slices per fold:')
    for fold in range(5):
        fold_df = df_slices[df_slices['fold'] == fold]
        wt = (fold_df['idh_label'] == 0).sum()
        mt = (fold_df['idh_label'] == 1).sum()
        print(f'  Fold {fold}: {len(fold_df):>5} slices — {wt} wildtype / {mt} mutant')
    print()
    print('Files in /kaggle/working/:')
    print('  labels.csv          — patient ID, IDH label, grade, fold')
    print('  patient_splits.json — fold assignments per patient')
    print('  slice_metadata.csv  — slice to patient mapping')
    print('  trackA_slices/      — 495 .npy files, float16, (n,7,224,224)')
    print('  Track B = trackA[:, :4, :, :] derived at runtime')
    

In [ ]:
if False: 
        """raise SystemExit("Tensors already saved to kaggle dataset with this cell")
    import subprocess, json, os, shutil
    
    WORKING = '/kaggle/working'
    SLICES_DIR = f'{WORKING}/trackA_slices'
    TENSORS_DS = 'adesaladaniel/glioma-idh-tensors'
    
    # 1. Clean up everything EXCEPT the slices to free every possible megabyte
    for folder in ['cnn_trackA', 'cnn_trackB', 'kaggle_sync', 'upload_batch']:
        path = os.path.join(WORKING, folder)
        if os.path.exists(path):
            shutil.rmtree(path)
    
    # 2. Check if metadata exists, if not, create it
    metadata_path = os.path.join(SLICES_DIR, 'dataset-metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump({
            'title': 'Glioma IDH Tensors',
            'id': TENSORS_DS,
            'licenses': [{'name': 'other'}],
            'isPrivate': True
        }, f)
    
    print("🚀 Starting the BIG upload using CLI TAR mode...")
    print("This will take a while. Do not close this tab.")
    
    # 3. Use TAR mode - it is much more stable for 20GB+ datasets
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', 
         '-p', SLICES_DIR, 
         '-m', 'Full 0.1 percent threshold tensors', 
         '--dir-mode', 'tar'], 
        capture_output=True, text=True
    )
    
    if result.returncode == 0:
        print("✅ SUCCESS! Tensors are now permanent on Kaggle.")
        print(result.stdout)
    else:
        print("❌ CLI Upload failed.")
        print("Error details:", result.stderr)
    """

In [ ]:
# ── TENSOR UPLOAD — COMPLETED ─────────────────────────────────────────────────
# All 495 preprocessed tensor files were uploaded to Kaggle on 2026-06-26.
#
# Permanent storage dataset:
#   adesaladaniel/glioma-idh-tensors
#
# Contents:
#   495 .npy files — one per patient
#   Shape per file: (n_slices, 7, 224, 224) float16
#   Threshold: 0.1% tumor coverage
#   Total slices: 28,881
#   Total size: ~20 GB
#
# To access in future sessions:
#   Use new Cell 4 (downloader version) — downloads in 2-3 minutes
#   No reprocessing ever needed again
#
# Upload method: kaggle datasets version --dir-mode tar
# Upload date: 2026-06-26
# Uploaded by: Daniel Adesala
print("Tensors permanently stored at: adesaladaniel/glioma-idh-tensors ✅")

In [4]:
# ── CELL 4: VERIFY TENSOR DATASET IS MOUNTED ─────────────────────────────────
# Tensors are permanently stored at:
#   adesaladaniel/glioma-idh-tensors
#
# Mounted as input dataset at:
#   /kaggle/input/glioma-idh-tensors/
#
# This uses ZERO working directory disk space.
# No preprocessing needed. No download needed.
# Just verify the mount and proceed to Cell 5.

import os
import numpy as np
import pandas as pd
import shutil

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'

print('='*60)
print('CELL 4 — TENSOR VERIFICATION')
print('='*60)

# ── CHECK TENSORS ARE ACCESSIBLE ──────────────────────────────────────────────
if not os.path.exists(SLICES_DIR):
    print('❌ ERROR: Tensor dataset not mounted.')
    print('   Go to notebook settings → Add Data')
    print('   Search for: adesaladaniel/glioma-idh-tensors')
    print('   Add it as input dataset and restart session.')
    raise SystemExit

npy_files = sorted([
    f for f in os.listdir(SLICES_DIR)
    if f.endswith('.npy')
])

print(f'Tensor location : {SLICES_DIR}')
print(f'Tensors found   : {len(npy_files)}/495')

# ── SPOT CHECK FIRST FILE ─────────────────────────────────────────────────────
if npy_files:
    sample_path = os.path.join(SLICES_DIR, npy_files[0])
    sample = np.load(sample_path, mmap_mode='r')
    print(f'Sample file     : {npy_files[0]}')
    print(f'Sample shape    : {sample.shape}')
    print(f'Sample dtype    : {sample.dtype}')
    print(f'Channels        : {sample.shape[1]} (expect 7 for Track A)')

# ── SPOT CHECK LAST FILE ──────────────────────────────────────────────────────
if len(npy_files) > 1:
    last_path = os.path.join(SLICES_DIR, npy_files[-1])
    last = np.load(last_path, mmap_mode='r')
    print(f'Last file       : {npy_files[-1]}')
    print(f'Last shape      : {last.shape}')

# ── DISK USAGE ────────────────────────────────────────────────────────────────
print()
total, used, free = shutil.disk_usage(WORKING)
print(f'Working directory disk usage:')
print(f'  Total : {total/1e9:.1f} GB')
print(f'  Used  : {used/1e9:.2f} GB')
print(f'  Free  : {free/1e9:.2f} GB')

# ── FINAL VERDICT ─────────────────────────────────────────────────────────────
print()
if len(npy_files) == 495:
    print('✅ All 495 tensors accessible')
    print('✅ Zero working directory space used for tensors')
    print('✅ Ready for Cell 5')
else:
    print(f'⚠️  Only {len(npy_files)}/495 tensors found')
    print('Check that glioma-idh-tensors is properly mounted')

CELL 4 — TENSOR VERIFICATION
Tensor location : /kaggle/input/datasets/adesaladaniel/glioma-idh-tensors
Tensors found   : 495/495
Sample file     : UCSF-PDGM-0004.npy
Sample shape    : (42, 7, 224, 224)
Sample dtype    : float16
Channels        : 7 (expect 7 for Track A)
Last file       : UCSF-PDGM-0541.npy
Last shape      : (82, 7, 224, 224)

Working directory disk usage:
  Total : 21.0 GB
  Used  : 0.00 GB
  Free  : 20.94 GB

✅ All 495 tensors accessible
✅ Zero working directory space used for tensors
✅ Ready for Cell 5


In [ ]:
# ── CELL 4 AUDIT: FULL TENSOR INTEGRITY CHECK ────────────────────────────────
# Verifies:
#   - All 495 patients present
#   - Slice counts match expected (~28,881 total)
#   - Fold distribution matches patient_splits.json
#   - IDH label alignment correct
#   - No corrupted files

import os
import json
import numpy as np
import pandas as pd
import shutil

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'

print('='*60)
print('CELL 4 AUDIT — TENSOR INTEGRITY CHECK')
print('='*60)

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map = dict(zip(df_labels['patient_id'],
                     df_labels['IDH_binary'].astype(int)))
fold_map  = dict(zip(df_labels['patient_id'],
                     df_labels['fold'].astype(int)))

# ── COUNT FILES ───────────────────────────────────────────────────────────────
npy_files = sorted([
    f for f in os.listdir(SLICES_DIR)
    if f.endswith('.npy')
])
patient_ids_on_disk = set(f.replace('.npy', '') for f in npy_files)
patient_ids_in_labels = set(df_labels['patient_id'].tolist())

missing_from_disk   = patient_ids_in_labels - patient_ids_on_disk
extra_on_disk       = patient_ids_on_disk - patient_ids_in_labels

print(f'\nFile count check:')
print(f'  Files on disk    : {len(npy_files)}')
print(f'  Patients in CSV  : {len(df_labels)}')
print(f'  Missing from disk: {len(missing_from_disk)}')
print(f'  Extra on disk    : {len(extra_on_disk)}')

if missing_from_disk:
    print(f'  Missing patients : {sorted(missing_from_disk)[:5]}')

# ── SLICE COUNT AND FOLD DISTRIBUTION ─────────────────────────────────────────
total_slices = 0
fold_counts  = {fold: {'wt': 0, 'mt': 0, 'total': 0} for fold in range(5)}
shape_issues = []
dtype_issues = []

print(f'\nChecking all {len(npy_files)} tensor files...')

for npy_file in npy_files:
    pid  = npy_file.replace('.npy', '')
    path = os.path.join(SLICES_DIR, npy_file)

    try:
        arr = np.load(path, mmap_mode='r')
    except Exception as e:
        print(f'  ❌ Corrupt file: {npy_file} — {e}')
        continue

    n_slices = arr.shape[0]
    total_slices += n_slices

    # Check shape
    if len(arr.shape) != 4 or arr.shape[1] != 7 or arr.shape[2] != 224:
        shape_issues.append((npy_file, arr.shape))

    # Check dtype
    if arr.dtype != np.float16:
        dtype_issues.append((npy_file, arr.dtype))

    # Fold distribution
    if pid in fold_map:
        fold = fold_map[pid]
        fold_counts[fold]['total'] += n_slices
        if label_map.get(pid, 0) == 0:
            fold_counts[fold]['wt'] += n_slices
        else:
            fold_counts[fold]['mt'] += n_slices

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print()
print('='*60)
print('AUDIT RESULTS')
print('='*60)
print(f'Patients    : {len(npy_files)}/495')
print(f'Total slices: {total_slices}')
print(f'Avg slices  : {total_slices/max(len(npy_files),1):.1f} per patient')
print()

print('Slices per fold:')
for fold in range(5):
    fc = fold_counts[fold]
    print(f'  Fold {fold}: {fc["total"]:>5} slices — '
          f'{fc["wt"]} wildtype / {fc["mt"]} mutant')

print()
print('Shape issues  :', len(shape_issues))
if shape_issues:
    for f, s in shape_issues[:3]:
        print(f'  {f}: {s}')

print('Dtype issues  :', len(dtype_issues))
if dtype_issues:
    for f, d in dtype_issues[:3]:
        print(f'  {f}: {d}')

print()
total_disk, used_disk, free_disk = shutil.disk_usage(WORKING)
print(f'Disk: {used_disk/1e9:.2f} GB used / '
      f'{total_disk/1e9:.1f} GB total / '
      f'{free_disk/1e9:.2f} GB free')

# ── FINAL VERDICT ─────────────────────────────────────────────────────────────
print()
print('='*60)
all_good = (
    len(npy_files) == 495 and
    len(shape_issues) == 0 and
    len(dtype_issues) == 0 and
    len(missing_from_disk) == 0
)

if all_good:
    print('✅ ALL CHECKS PASSED')
    print(f'✅ {len(npy_files)} patients')
    print(f'✅ {total_slices} slices')
    print(f'✅ All shapes correct (n, 7, 224, 224)')
    print(f'✅ All dtypes correct (float16)')
    print(f'✅ {free_disk/1e9:.2f} GB free for model training')
    print()
    print('Ready for Cell 5 🚀')
else:
    print('⚠️  SOME CHECKS FAILED — review issues above')

## Cell 5: CNN Training

**Track A (ResNet18 7-channel)**

In [9]:
# ── CELL 5 (FIXED): CNN TRAINING — TRACK A (ResNet18 7-channel) ──────────────
# FIXED: saves each fold's .pth directly to Kaggle immediately
# so weights are never lost on session timeout

import os, json, shutil, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
import torchvision.transforms.functional as TF
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from kaggle_secrets import UserSecretsClient

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

os.makedirs(f'{WORKING}/cnn_trackA', exist_ok=True)

# ── KAGGLE API ────────────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
print('Kaggle API ready')

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 7
N_CLASSES    = 2
BATCH_SIZE   = 32
MAX_EPOCHS   = 50
PATIENCE     = 10
LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-4
WEIGHT_DECAY = 1e-4
CLASS_WEIGHT = 3.81
SEED         = 42
N_FOLDS      = 5
SAVE_DIR     = f'{WORKING}/cnn_trackA'

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Device   : {DEVICE}')
print(f'Track    : A — {N_CHANNELS} channels')

# ── RESTORE EXISTING PROGRESS FROM KAGGLE ────────────────────────────────────
print('\nChecking Kaggle for existing CNN Track A progress...')
sync_dir = f'{WORKING}/sync_trackA'
os.makedirs(sync_dir, exist_ok=True)

sync_result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

completed_folds = []
fold_results    = {}
all_val_preds   = {}

if sync_result.returncode == 0:
    # Restore progress JSON
    for fname in os.listdir(sync_dir):
        src = os.path.join(sync_dir, fname)
        dst = os.path.join(WORKING, fname)
        shutil.copy(src, dst)

    progress_path = f'{WORKING}/cnn_trackA_progress.json'
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            prog = json.load(f)
        completed_folds = prog.get('completed_folds', [])
        fold_results    = prog.get('fold_results', {})
        all_val_preds   = prog.get('all_val_preds', {})
        print(f'Resuming — completed folds: {completed_folds}')
    else:
        print('No existing progress — starting fresh')

    # Restore weights for completed folds
    for fold in completed_folds:
        kaggle_name = f'cnn_trackA_fold{fold}_best.pth'
        src = os.path.join(sync_dir, kaggle_name)
        dst = os.path.join(SAVE_DIR, f'fold{fold}_best.pth')
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  Restored weights: fold {fold}')
else:
    print('No existing data on Kaggle — starting fresh')

shutil.rmtree(sync_dir, ignore_errors=True)

# ── SAFE COMMIT — UPLOADS EVERYTHING ─────────────────────────────────────────
def commit_to_kaggle(message):
    """
    Commits ALL cnn_trackA files plus any existing
    radiomics files — cumulative, never overwrites.
    """
    commit_dir = f'{WORKING}/commit_dir'
    os.makedirs(commit_dir, exist_ok=True)

    with open(os.path.join(commit_dir,
                           'dataset-metadata.json'), 'w') as f:
        json.dump({'title':     'Glioma IDH Models',
                   'id':        MODELS_DATASET,
                   'licenses':  [{'name': 'other'}],
                   'isPrivate': True}, f)

    # Include ALL model files — track A, track B, radiomics
    for fname in os.listdir(WORKING):
        src = os.path.join(WORKING, fname)
        lnk = os.path.join(commit_dir, fname)
        if any(fname.startswith(p) for p in
               ['cnn_track', 'radiomics_', 'fusion_']) and \
           any(fname.endswith(e) for e in
               ['.pth', '.json', '.csv']):
            if os.path.exists(src) and not os.path.lexists(lnk):
                os.symlink(os.path.abspath(src), lnk)

    # Also copy labels and splits so they survive resets
    for fname in ['labels.csv', 'patient_splits.json']:
        src = os.path.join(WORKING, fname)
        lnk = os.path.join(commit_dir, fname)
        if os.path.exists(src) and not os.path.lexists(lnk):
            os.symlink(os.path.abspath(src), lnk)

    r = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', commit_dir, '-m', message,
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
    status = '✅' if r.returncode == 0 else '⚠️'
    print(f'  {status} Kaggle commit: {message}')
    if r.returncode != 0:
        print(f'     Error: {r.stderr[:200]}')

    shutil.rmtree(commit_dir, ignore_errors=True)
    return r.returncode == 0

# ── DATASET ───────────────────────────────────────────────────────────────────
class GliomaDataset(Dataset):
    def __init__(self, patient_ids, df_labels, slices_dir,
                 n_channels=7, augment=False):
        self.slices_dir = slices_dir
        self.n_channels = n_channels
        self.augment    = augment
        self.items      = []
        label_map = dict(zip(df_labels['patient_id'],
                             df_labels['IDH_binary'].astype(int)))
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            arr = np.load(npy_path, mmap_mode='r')
            for s in range(arr.shape[0]):
                self.items.append((pid, s, label_map[pid]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s, label = self.items[idx]
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            if torch.rand(1) > 0.5:
                angle = (torch.rand(1).item() - 0.5) * 30
                x = TF.rotate(x, angle)
            x = x * (0.9 + torch.rand(1).item() * 0.2)
            x = x + torch.randn_like(x) * 0.02
        return x, label, pid

# ── MODEL ─────────────────────────────────────────────────────────────────────
def build_model(n_channels, n_classes):
    model = resnet18(weights='IMAGENET1K_V1')
    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    with torch.no_grad():
        avg_w = model.conv1.weight.mean(dim=1, keepdim=True)
        new_conv.weight = nn.Parameter(
            avg_w.repeat(1, n_channels, 1, 1))
    model.conv1 = new_conv
    model.fc = nn.Sequential(
        nn.Linear(512, 256), nn.ReLU(),
        nn.Dropout(0.5), nn.Linear(256, n_classes)
    )
    return model

# ── TRAIN ONE EPOCH ───────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

# ── EVALUATE ──────────────────────────────────────────────────────────────────
def evaluate(model, loader, criterion, device, label_map):
    model.eval()
    total_loss, patient_probs = 0, {}
    all_slice_probs, all_slice_labels = [], []

    with torch.no_grad():
        for x, y, pids in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            probs  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            labels = y.cpu().numpy()
            all_slice_probs.extend(probs)
            all_slice_labels.extend(labels)
            for prob, pid in zip(probs, pids):
                patient_probs.setdefault(pid, []).append(float(prob))

    pid_list       = list(patient_probs.keys())
    pid_mean_probs = [np.mean(patient_probs[p]) for p in pid_list]
    pid_labels     = [label_map[p] for p in pid_list]
    pid_preds      = [1 if p >= 0.5 else 0 for p in pid_mean_probs]

    patient_auc = roc_auc_score(pid_labels, pid_mean_probs)
    slice_auc   = roc_auc_score(all_slice_labels, all_slice_probs)

    cm = confusion_matrix(pid_labels, pid_preds, labels=[0, 1])
    tn, fp, fn, tp = (cm.ravel() if cm.shape == (2, 2)
                      else (0, 0, 0, 0))

    metrics = {
        'patient_auc': round(float(patient_auc), 4),
        'slice_auc':   round(float(slice_auc), 4),
        'accuracy':    round(float(accuracy_score(
            pid_labels, pid_preds)), 4),
        'sensitivity': round(float(recall_score(
            pid_labels, pid_preds, zero_division=0)), 4),
        'specificity': round(float(
            tn / (tn + fp) if (tn + fp) > 0 else 0.0), 4),
        'f1':          round(float(f1_score(
            pid_labels, pid_preds, zero_division=0)), 4),
        'brier':       round(float(brier_score_loss(
            pid_labels, pid_mean_probs)), 4),
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    }
    return total_loss / len(loader.dataset), metrics, patient_probs

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map     = dict(zip(df_labels['patient_id'],
                         df_labels['IDH_binary'].astype(int)))
class_weights = torch.tensor([1.0, CLASS_WEIGHT],
                              dtype=torch.float32).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# ── 5-FOLD TRAINING LOOP ──────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK A — 5-FOLD CROSS VALIDATION')
print(f'{"="*60}')

for fold in range(N_FOLDS):
    if fold in completed_folds:
        r = fold_results[str(fold)]
        print(f'\nFold {fold} — already done '
              f'(AUC {r["best_auc"]:.4f}), skipping')
        continue

    print(f'\nFOLD {fold}/{N_FOLDS-1}')
    print('-'*40)

    val_patients   = splits[f'fold_{fold}']
    train_patients = [p for f2 in range(N_FOLDS) if f2 != fold
                      for p in splits[f'fold_{f2}']]

    train_ds = GliomaDataset(
        train_patients, df_labels, SLICES_DIR,
        n_channels=N_CHANNELS, augment=True)
    val_ds = GliomaDataset(
        val_patients, df_labels, SLICES_DIR,
        n_channels=N_CHANNELS, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True)

    print(f'Train: {len(train_patients)} patients / '
          f'{len(train_ds)} slices')
    print(f'Val  : {len(val_patients)} patients / '
          f'{len(val_ds)} slices')

    model = build_model(N_CHANNELS, N_CLASSES).to(DEVICE)

    # Freeze backbone for first 5 epochs
    for name, param in model.named_parameters():
        if not name.startswith('fc'):
            param.requires_grad = False

    optimizer = optim.AdamW([
        {'params': model.fc.parameters(), 'lr': LR_HEAD},
        {'params': [p for n, p in model.named_parameters()
                    if not n.startswith('fc')],
         'lr': LR_BACKBONE}
    ], weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_auc = 0.0
    best_epoch   = 0
    patience_ctr = 0
    best_metrics = {}
    history      = []

    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == 6:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW([
                {'params': model.fc.parameters(),
                 'lr': LR_HEAD},
                {'params': [p for n, p in
                            model.named_parameters()
                            if not n.startswith('fc')],
                 'lr': LR_BACKBONE}
            ], weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(
                optimizer, T_max=MAX_EPOCHS - 5)
            print(f'  Epoch {epoch}: backbone unfrozen')

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE)
        val_loss, metrics, _ = evaluate(
            model, val_loader, criterion, DEVICE, label_map)
        scheduler.step()

        history.append({
            'epoch': epoch,
            'train_loss': round(float(train_loss), 4),
            'val_loss':   round(float(val_loss), 4),
            **metrics
        })

        print(f'  Epoch {epoch:02d} | '
              f'loss {train_loss:.4f}/{val_loss:.4f} | '
              f'pt_AUC {metrics["patient_auc"]:.4f} | '
              f'sens {metrics["sensitivity"]:.4f} | '
              f'spec {metrics["specificity"]:.4f} | '
              f'F1 {metrics["f1"]:.4f}')

        if metrics['patient_auc'] > best_val_auc:
            best_val_auc = metrics['patient_auc']
            best_epoch   = epoch
            best_metrics = metrics.copy()
            patience_ctr = 0
            torch.save(model.state_dict(),
                       f'{SAVE_DIR}/fold{fold}_best.pth')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  Early stopping at epoch {epoch} '
                      f'(best epoch {best_epoch}, '
                      f'AUC {best_val_auc:.4f})')
                break

    # Reload best weights and get final predictions
    model.load_state_dict(torch.load(
        f'{SAVE_DIR}/fold{fold}_best.pth',
        map_location=DEVICE))
    _, best_metrics, patient_probs = evaluate(
        model, val_loader, criterion, DEVICE, label_map)

    for pid, probs in patient_probs.items():
        all_val_preds[pid] = {
            'mean_prob': float(np.mean(probs)),
            'label':     int(label_map[pid]),
            'fold':      fold
        }

    fold_results[str(fold)] = {
        'best_epoch':   best_epoch,
        'best_auc':     round(best_val_auc, 4),
        'best_metrics': best_metrics,
        'history':      history
    }
    completed_folds.append(fold)

    print(f'\n  Fold {fold} Summary:')
    print(f'    AUC         : {best_metrics["patient_auc"]:.4f}')
    print(f'    Sensitivity : {best_metrics["sensitivity"]:.4f}')
    print(f'    Specificity : {best_metrics["specificity"]:.4f}')
    print(f'    F1          : {best_metrics["f1"]:.4f}')

    # ── SAVE PROGRESS JSON ────────────────────────────────────────────────────
    progress = {
        'completed_folds': completed_folds,
        'fold_results':    fold_results,
        'all_val_preds':   all_val_preds
    }
    with open(f'{WORKING}/cnn_trackA_progress.json', 'w') as f:
        json.dump(progress, f)

    # Copy weights to working root for commit
    shutil.copy(
        f'{SAVE_DIR}/fold{fold}_best.pth',
        f'{WORKING}/cnn_trackA_fold{fold}_best.pth'
    )

    # ── IMMEDIATE KAGGLE COMMIT AFTER EVERY FOLD ──────────────────────────────
    commit_to_kaggle(
        f'Track A fold {fold} — AUC {best_val_auc:.4f}')

    del model, optimizer, scheduler, train_ds, val_ds
    del train_loader, val_loader
    torch.cuda.empty_cache()

    _, used, free = shutil.disk_usage(WORKING)
    print(f'  Disk: {used/1e9:.2f} GB used / '
          f'{free/1e9:.2f} GB free')

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK A — COMPLETE')
print(f'{"="*60}')

aucs = [fold_results[str(f)]['best_auc'] for f in range(N_FOLDS)]
sens = [fold_results[str(f)]['best_metrics']['sensitivity']
        for f in range(N_FOLDS)]
spec = [fold_results[str(f)]['best_metrics']['specificity']
        for f in range(N_FOLDS)]
f1s  = [fold_results[str(f)]['best_metrics']['f1']
        for f in range(N_FOLDS)]

print(f'\nPer-fold AUC : {[round(a,4) for a in aucs]}')
print(f'Mean AUC     : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'Mean Sens    : {np.mean(sens):.4f} ± {np.std(sens):.4f}')
print(f'Mean Spec    : {np.mean(spec):.4f} ± {np.std(spec):.4f}')
print(f'Mean F1      : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'\nOOF preds    : {len(all_val_preds)} patients')
print(f'\n✅ Ready for Cell 6 — CNN Track B')

Kaggle API ready
Device   : cuda
Track    : A — 7 channels

Checking Kaggle for existing CNN Track A progress...
Resuming — completed folds: [0, 1, 2, 3, 4]

CNN TRACK A — 5-FOLD CROSS VALIDATION

Fold 0 — already done (AUC 0.9709), skipping

Fold 1 — already done (AUC 0.9411), skipping

Fold 2 — already done (AUC 0.9554), skipping

Fold 3 — already done (AUC 0.9982), skipping

Fold 4 — already done (AUC 0.9750), skipping

CNN TRACK A — COMPLETE

Per-fold AUC : [0.9709, 0.9411, 0.9554, 0.9982, 0.975]
Mean AUC     : 0.9681 ± 0.0192
Mean Sens    : 0.7662 ± 0.0906
Mean Spec    : 0.9489 ± 0.0354
Mean F1      : 0.7817 ± 0.0345

OOF preds    : 495 patients

✅ Ready for Cell 6 — CNN Track B


## Cell 6: CNN Training

**Track B (ResNet18 4-channel)**

In [ ]:
# ── CELL 6 (FIXED): CNN TRAINING — TRACK B (ResNet18 4-channel) ──────────────
# FIXED: syncs from Kaggle first so it never overwrites existing files
# FIXED: commits immediately after every fold with ALL files included
# Track B = first 4 channels: T1, T1c, T2, FLAIR

import os, json, shutil, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
import torchvision.transforms.functional as TF
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from kaggle_secrets import UserSecretsClient

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

os.makedirs(f'{WORKING}/cnn_trackB', exist_ok=True)

# ── KAGGLE API ────────────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
print('Kaggle API ready')

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 4
N_CLASSES    = 2
BATCH_SIZE   = 32
MAX_EPOCHS   = 50
PATIENCE     = 10
LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-4
WEIGHT_DECAY = 1e-4
CLASS_WEIGHT = 3.81
SEED         = 42
N_FOLDS      = 5
SAVE_DIR     = f'{WORKING}/cnn_trackB'

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Device   : {DEVICE}')
print(f'Track    : B — {N_CHANNELS} channels (T1, T1c, T2, FLAIR)')

# ── STEP 1: SYNC EVERYTHING FROM KAGGLE FIRST ────────────────────────────────
# This is the critical fix — download ALL existing files before saving anything
# so we never accidentally overwrite radiomics or Track A files
print('\n' + '='*60)
print('SYNCING ALL EXISTING FILES FROM KAGGLE FIRST')
print('='*60)

sync_dir = f'{WORKING}/sync_trackB'
os.makedirs(sync_dir, exist_ok=True)

sync_result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

completed_folds = []
fold_results    = {}
all_val_preds   = {}

if sync_result.returncode == 0:
    synced_files = os.listdir(sync_dir)
    print(f'Downloaded {len(synced_files)} existing files:')

    for fname in synced_files:
        src = os.path.join(sync_dir, fname)
        dst = os.path.join(WORKING, fname)
        shutil.copy(src, dst)
        print(f'  ✅ {fname}')

        # Route Track B weights to subdirectory
        if fname.startswith('cnn_trackB') and \
           fname.endswith('.pth'):
            fold_dst = os.path.join(
                SAVE_DIR,
                fname.replace('cnn_trackB_', ''))
            shutil.copy(src, fold_dst)

    # Load Track B progress if it exists
    progress_path = f'{WORKING}/cnn_trackB_progress.json'
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            prog = json.load(f)
        completed_folds = prog.get('completed_folds', [])
        fold_results    = prog.get('fold_results', {})
        all_val_preds   = prog.get('all_val_preds', {})
        print(f'\nResuming Track B — '
              f'completed folds: {completed_folds}')
    else:
        print('\nNo Track B progress found — starting fresh')

else:
    print('No existing files on Kaggle — starting fresh')

shutil.rmtree(sync_dir, ignore_errors=True)

print(f'\nFiles now in /kaggle/working/:')
all_working = [f for f in os.listdir(WORKING)
               if any(f.endswith(e)
                      for e in ['.pth', '.json', '.csv'])]
for f in sorted(all_working):
    print(f'  {f}')

# ── CUMULATIVE COMMIT — SAVES EVERYTHING ALWAYS ───────────────────────────────
def commit_to_kaggle(message):
    """
    Commits ALL files from working directory —
    Track A weights, Track B weights, radiomics,
    labels, splits — everything in one version.
    Nothing gets lost.
    """
    commit_dir = f'{WORKING}/commit_dir'
    os.makedirs(commit_dir, exist_ok=True)

    with open(os.path.join(commit_dir,
                           'dataset-metadata.json'), 'w') as f:
        json.dump({'title':     'Glioma IDH Models',
                   'id':        MODELS_DATASET,
                   'licenses':  [{'name': 'other'}],
                   'isPrivate': True}, f)

    files_linked = []
    for fname in os.listdir(WORKING):
        src = os.path.join(WORKING, fname)
        lnk = os.path.join(commit_dir, fname)

        # Include ALL relevant file types
        should_include = (
            any(fname.startswith(p) for p in [
                'cnn_track',    # CNN weights + progress
                'radiomics_',   # Radiomics CSVs + results
                'fusion_',      # Future fusion results
            ]) and
            any(fname.endswith(e) for e in [
                '.pth', '.json', '.csv'
            ])
        ) or fname in [
            'labels.csv',           # Always include
            'patient_splits.json'   # Always include
        ]

        if should_include and os.path.exists(src) and \
           not os.path.lexists(lnk):
            os.symlink(os.path.abspath(src), lnk)
            files_linked.append(fname)

    print(f'  Committing {len(files_linked)} files...')

    r = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', commit_dir, '-m', message,
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )

    status = '✅' if r.returncode == 0 else '⚠️'
    print(f'  {status} {message}')
    if r.returncode != 0:
        print(f'     Error: {r.stderr[:300]}')

    shutil.rmtree(commit_dir, ignore_errors=True)
    return r.returncode == 0

# ── DATASET ───────────────────────────────────────────────────────────────────
class GliomaDataset(Dataset):
    def __init__(self, patient_ids, df_labels, slices_dir,
                 n_channels=4, augment=False):
        self.slices_dir = slices_dir
        self.n_channels = n_channels
        self.augment    = augment
        self.items      = []
        label_map = dict(zip(df_labels['patient_id'],
                             df_labels['IDH_binary'].astype(int)))
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            arr = np.load(npy_path, mmap_mode='r')
            for s in range(arr.shape[0]):
                self.items.append((pid, s, label_map[pid]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s, label = self.items[idx]
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            if torch.rand(1) > 0.5:
                angle = (torch.rand(1).item() - 0.5) * 30
                x = TF.rotate(x, angle)
            x = x * (0.9 + torch.rand(1).item() * 0.2)
            x = x + torch.randn_like(x) * 0.02
        return x, label, pid

# ── MODEL ─────────────────────────────────────────────────────────────────────
def build_model(n_channels, n_classes):
    model = resnet18(weights='IMAGENET1K_V1')
    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    with torch.no_grad():
        avg_w = model.conv1.weight.mean(dim=1, keepdim=True)
        new_conv.weight = nn.Parameter(
            avg_w.repeat(1, n_channels, 1, 1))
    model.conv1 = new_conv
    model.fc = nn.Sequential(
        nn.Linear(512, 256), nn.ReLU(),
        nn.Dropout(0.5), nn.Linear(256, n_classes)
    )
    return model

# ── TRAIN ONE EPOCH ───────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

# ── EVALUATE ──────────────────────────────────────────────────────────────────
def evaluate(model, loader, criterion, device, label_map):
    model.eval()
    total_loss, patient_probs = 0, {}
    all_slice_probs, all_slice_labels = [], []

    with torch.no_grad():
        for x, y, pids in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            probs  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            labels = y.cpu().numpy()
            all_slice_probs.extend(probs)
            all_slice_labels.extend(labels)
            for prob, pid in zip(probs, pids):
                patient_probs.setdefault(
                    pid, []).append(float(prob))

    pid_list       = list(patient_probs.keys())
    pid_mean_probs = [np.mean(patient_probs[p])
                      for p in pid_list]
    pid_labels     = [label_map[p] for p in pid_list]
    pid_preds      = [1 if p >= 0.5 else 0
                      for p in pid_mean_probs]

    patient_auc = roc_auc_score(pid_labels, pid_mean_probs)
    slice_auc   = roc_auc_score(all_slice_labels,
                                all_slice_probs)

    cm = confusion_matrix(pid_labels, pid_preds, labels=[0, 1])
    tn, fp, fn, tp = (cm.ravel() if cm.shape == (2, 2)
                      else (0, 0, 0, 0))

    metrics = {
        'patient_auc': round(float(patient_auc), 4),
        'slice_auc':   round(float(slice_auc), 4),
        'accuracy':    round(float(accuracy_score(
            pid_labels, pid_preds)), 4),
        'sensitivity': round(float(recall_score(
            pid_labels, pid_preds, zero_division=0)), 4),
        'specificity': round(float(
            tn / (tn + fp) if (tn + fp) > 0 else 0.0), 4),
        'f1':          round(float(f1_score(
            pid_labels, pid_preds, zero_division=0)), 4),
        'brier':       round(float(brier_score_loss(
            pid_labels, pid_mean_probs)), 4),
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    }
    return (total_loss / len(loader.dataset),
            metrics, patient_probs)

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map     = dict(zip(df_labels['patient_id'],
                         df_labels['IDH_binary'].astype(int)))
class_weights = torch.tensor([1.0, CLASS_WEIGHT],
                              dtype=torch.float32).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# ── 5-FOLD TRAINING LOOP ──────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK B — 5-FOLD CROSS VALIDATION')
print(f'{"="*60}')

for fold in range(N_FOLDS):
    if fold in completed_folds:
        r = fold_results[str(fold)]
        print(f'\nFold {fold} — already done '
              f'(AUC {r["best_auc"]:.4f}), skipping')
        continue

    print(f'\nFOLD {fold}/{N_FOLDS-1}')
    print('-'*40)

    val_patients   = splits[f'fold_{fold}']
    train_patients = [p for f2 in range(N_FOLDS)
                      if f2 != fold
                      for p in splits[f'fold_{f2}']]

    train_ds = GliomaDataset(
        train_patients, df_labels, SLICES_DIR,
        n_channels=N_CHANNELS, augment=True)
    val_ds = GliomaDataset(
        val_patients, df_labels, SLICES_DIR,
        n_channels=N_CHANNELS, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True)

    print(f'Train: {len(train_patients)} patients / '
          f'{len(train_ds)} slices')
    print(f'Val  : {len(val_patients)} patients / '
          f'{len(val_ds)} slices')

    model = build_model(N_CHANNELS, N_CLASSES).to(DEVICE)

    # Freeze backbone for first 5 epochs
    for name, param in model.named_parameters():
        if not name.startswith('fc'):
            param.requires_grad = False

    optimizer = optim.AdamW([
        {'params': model.fc.parameters(),
         'lr': LR_HEAD},
        {'params': [p for n, p in model.named_parameters()
                    if not n.startswith('fc')],
         'lr': LR_BACKBONE}
    ], weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_auc = 0.0
    best_epoch   = 0
    patience_ctr = 0
    best_metrics = {}
    history      = []

    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == 6:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW([
                {'params': model.fc.parameters(),
                 'lr': LR_HEAD},
                {'params': [p for n, p in
                            model.named_parameters()
                            if not n.startswith('fc')],
                 'lr': LR_BACKBONE}
            ], weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(
                optimizer, T_max=MAX_EPOCHS - 5)
            print(f'  Epoch {epoch}: backbone unfrozen')

        train_loss = train_one_epoch(
            model, train_loader, optimizer,
            criterion, DEVICE)
        val_loss, metrics, _ = evaluate(
            model, val_loader, criterion,
            DEVICE, label_map)
        scheduler.step()

        history.append({
            'epoch':      epoch,
            'train_loss': round(float(train_loss), 4),
            'val_loss':   round(float(val_loss), 4),
            **metrics
        })

        print(f'  Epoch {epoch:02d} | '
              f'loss {train_loss:.4f}/{val_loss:.4f} | '
              f'pt_AUC {metrics["patient_auc"]:.4f} | '
              f'sens {metrics["sensitivity"]:.4f} | '
              f'spec {metrics["specificity"]:.4f} | '
              f'F1 {metrics["f1"]:.4f}')

        if metrics['patient_auc'] > best_val_auc:
            best_val_auc = metrics['patient_auc']
            best_epoch   = epoch
            best_metrics = metrics.copy()
            patience_ctr = 0
            torch.save(
                model.state_dict(),
                f'{SAVE_DIR}/fold{fold}_best.pth')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  Early stopping at epoch {epoch} '
                      f'(best epoch {best_epoch}, '
                      f'AUC {best_val_auc:.4f})')
                break

    # Reload best weights for final evaluation
    model.load_state_dict(torch.load(
        f'{SAVE_DIR}/fold{fold}_best.pth',
        map_location=DEVICE))
    _, best_metrics, patient_probs = evaluate(
        model, val_loader, criterion, DEVICE, label_map)

    for pid, probs in patient_probs.items():
        all_val_preds[pid] = {
            'mean_prob': float(np.mean(probs)),
            'label':     int(label_map[pid]),
            'fold':      fold
        }

    fold_results[str(fold)] = {
        'best_epoch':   best_epoch,
        'best_auc':     round(best_val_auc, 4),
        'best_metrics': best_metrics,
        'history':      history
    }
    completed_folds.append(fold)

    print(f'\n  Fold {fold} Summary:')
    print(f'    AUC         : {best_metrics["patient_auc"]:.4f}')
    print(f'    Sensitivity : {best_metrics["sensitivity"]:.4f}')
    print(f'    Specificity : {best_metrics["specificity"]:.4f}')
    print(f'    F1          : {best_metrics["f1"]:.4f}')
    print(f'    TP:{best_metrics["tp"]} '
          f'TN:{best_metrics["tn"]} '
          f'FP:{best_metrics["fp"]} '
          f'FN:{best_metrics["fn"]}')

    # ── SAVE PROGRESS JSON ────────────────────────────────────────────────────
    with open(f'{WORKING}/cnn_trackB_progress.json', 'w') as f:
        json.dump({
            'completed_folds': completed_folds,
            'fold_results':    fold_results,
            'all_val_preds':   all_val_preds
        }, f)

    # Copy weights to working root for commit
    shutil.copy(
        f'{SAVE_DIR}/fold{fold}_best.pth',
        f'{WORKING}/cnn_trackB_fold{fold}_best.pth'
    )

    # ── IMMEDIATE CUMULATIVE COMMIT AFTER EVERY FOLD ──────────────────────────
    commit_to_kaggle(
        f'Track B fold {fold} — AUC {best_val_auc:.4f}')

    del model, optimizer, scheduler
    del train_ds, val_ds, train_loader, val_loader
    torch.cuda.empty_cache()

    _, used, free = shutil.disk_usage(WORKING)
    print(f'  Disk: {used/1e9:.2f} GB used / '
          f'{free/1e9:.2f} GB free')

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK B — COMPLETE')
print(f'{"="*60}')

aucs = [fold_results[str(f)]['best_auc']
        for f in range(N_FOLDS)]
sens = [fold_results[str(f)]['best_metrics']['sensitivity']
        for f in range(N_FOLDS)]
spec = [fold_results[str(f)]['best_metrics']['specificity']
        for f in range(N_FOLDS)]
f1s  = [fold_results[str(f)]['best_metrics']['f1']
        for f in range(N_FOLDS)]

print(f'\nPer-fold AUC : {[round(a,4) for a in aucs]}')
print(f'Mean AUC     : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'Mean Sens    : {np.mean(sens):.4f} ± {np.std(sens):.4f}')
print(f'Mean Spec    : {np.mean(spec):.4f} ± {np.std(spec):.4f}')
print(f'Mean F1      : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'\nOOF preds    : {len(all_val_preds)} patients')
print(f'\n✅ Track B complete — weights saved to Kaggle')
print(f'✅ All radiomics files preserved')
print(f'Next: Run Fixed Cell 5 for Track A')

## Cell 7: Radiomics feature extraction



In [ ]:
# ── CELL 7: RADIOMICS FEATURE EXTRACTION ─────────────────────────────────────
# Extracts PyRadiomics features from original NIfTI files
# Track A: ~900 features (7 sequences × feature classes + shape)
# Track B: ~550 features (4 sequences × feature classes + shape)
#
# Install : PyRadiomics via git (Python 3.12 compatible)
# Reads   : original NIfTI batch folders
# Saves   : radiomics_trackA.csv + radiomics_trackB.csv
# Commits : to adesaladaniel/glioma-idh-models
# Resume  : safe — skips completed patients automatically
# Time    : ~3-4 hours CPU

import os, sys, subprocess

# ── INSTALL PYRADIOMICS ───────────────────────────────────────────────────────
print('Installing PyRadiomics (git)...')
os.system('pip install "pyradiomics @ git+https://github.com/'
          'AIM-Harvard/pyradiomics.git" -q')

try:
    from radiomics import featureextractor
    import radiomics
    print(f'✅ PyRadiomics {radiomics.__version__} ready')
except ImportError as e:
    print(f'❌ PyRadiomics import failed: {e}')
    raise

import os, json, shutil, warnings
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import logging

warnings.filterwarnings('ignore')
logging.getLogger('radiomics').setLevel(logging.ERROR)
logging.getLogger('pykwalify').setLevel(logging.ERROR)

WORKING     = '/kaggle/working'
KAGGLE_USER = 'adesaladaniel'
BATCH_ROOTS = [
    f'/kaggle/input/datasets/{KAGGLE_USER}/ucsf-pdgm-batch-{i:02d}'
    for i in range(1, 11)
]

TRACK_A_SEQS   = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TRACK_B_SEQS   = ['T1', 'T1c', 'T2', 'FLAIR']
TUMOR_SEQ      = 'tumor_segmentation'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')

# ── REBUILD PATIENT DIRS ──────────────────────────────────────────────────────
patient_dirs = {}
for batch_root in BATCH_ROOTS:
    if not os.path.exists(batch_root):
        continue
    for entry in sorted(os.listdir(batch_root)):
        if entry.startswith('UCSF-PDGM-'):
            full_path = os.path.join(batch_root, entry)
            if os.path.isdir(full_path):
                patient_dirs[entry] = full_path

print(f'Patient folders : {len(patient_dirs)}/495')

# ── LOAD LABELS ───────────────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
label_map = dict(zip(df_labels['patient_id'],
                     df_labels['IDH_binary'].astype(int)))
fold_map  = dict(zip(df_labels['patient_id'],
                     df_labels['fold'].astype(int)))
print(f'Labels loaded   : {len(df_labels)} patients')

# ── NIfTI PATH HELPER ─────────────────────────────────────────────────────────
def get_nii_path(patient_dir, patient_id, seq_name):
    filename = f'{patient_id}_{seq_name}.nii'
    nested   = os.path.join(patient_dir, filename, filename)
    flat     = os.path.join(patient_dir, filename)
    if os.path.isfile(nested):
        return nested
    elif os.path.isfile(flat):
        return flat
    return None

# ── PYRADIOMICS EXTRACTOR ─────────────────────────────────────────────────────
def build_extractor():
    params = {
        'imageType': {
            'Original': {}
        },
        'featureClass': {
            'firstorder': [],
            'shape':      [],
            'glcm':       [],
            'glrlm':      [],
            'glszm':      []
        },
        'setting': {
            'binWidth':              25,
            'resampledPixelSpacing': [1, 1, 1],
            'interpolator':          'sitkBSpline',
            'padDistance':           10,
            'normalize':             True,
            'normalizeScale':        100,
            'removeOutliers':        3.0,
            'minimumROISize':        10,
            'geometryTolerance':     1e-6,
            'correctMask':           True
        }
    }
    return featureextractor.RadiomicsFeatureExtractor(params)

extractor = build_extractor()
print('Extractor configured ✅')

# ── FEATURE EXTRACTION FUNCTION ───────────────────────────────────────────────
def extract_features_for_patient(patient_id, patient_dir, sequences):
    try:
        # Load tumor mask
        tumor_path = get_nii_path(patient_dir, patient_id, TUMOR_SEQ)
        if tumor_path is None:
            return None, 'No tumor mask found'

        mask_sitk_orig = sitk.ReadImage(tumor_path)
        mask_arr       = sitk.GetArrayFromImage(mask_sitk_orig)
        mask_bin       = (mask_arr > 0).astype(np.uint8)

        if mask_bin.sum() < 10:
            return None, f'Tumor mask too small ({mask_bin.sum()} voxels)'

        mask_sitk = sitk.GetImageFromArray(mask_bin)
        mask_sitk.CopyInformation(mask_sitk_orig)

        all_features = {}

        for seq in sequences:
            seq_path = get_nii_path(patient_dir, patient_id, seq)
            if seq_path is None:
                return None, f'Missing sequence: {seq}'

            img_sitk = sitk.ReadImage(seq_path)
            img_sitk = sitk.Cast(img_sitk, sitk.sitkFloat32)

            result = extractor.execute(img_sitk, mask_sitk, label=1)

            for key, val in result.items():
                if key.startswith('original_'):
                    clean_key = key.replace('original_', f'{seq}_')
                    try:
                        all_features[clean_key] = float(val)
                    except (TypeError, ValueError):
                        pass

        return all_features, None

    except Exception as e:
        return None, str(e)

# ── LOAD OR INITIALISE PROGRESS ───────────────────────────────────────────────
progress_path_A = f'{WORKING}/radiomics_trackA_progress.json'
progress_path_B = f'{WORKING}/radiomics_trackB_progress.json'

# ── CHECK KAGGLE FOR EXISTING PROGRESS ───────────────────────────────────────
print('\nChecking Kaggle for existing radiomics progress...')
sync_dir = f'{WORKING}/rad_sync'
os.makedirs(sync_dir, exist_ok=True)

sync_result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

if sync_result.returncode == 0:
    for fname in ['radiomics_trackA_progress.json',
                  'radiomics_trackB_progress.json',
                  'radiomics_trackA.csv',
                  'radiomics_trackB.csv']:
        src = os.path.join(sync_dir, fname)
        dst = os.path.join(WORKING, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  Restored: {fname}')

shutil.rmtree(sync_dir, ignore_errors=True)

# Track A progress
if os.path.exists(progress_path_A):
    with open(progress_path_A) as f:
        prog_A = json.load(f)
    completed_A = set(prog_A.get('completed', []))
    failed_A    = prog_A.get('failed', {})
    rows_A      = prog_A.get('rows', [])
    print(f'Track A: resuming — {len(completed_A)} done, '
          f'{len(failed_A)} failed')
else:
    completed_A = set()
    failed_A    = {}
    rows_A      = []
    print('Track A: starting fresh')

# Track B progress
if os.path.exists(progress_path_B):
    with open(progress_path_B) as f:
        prog_B = json.load(f)
    completed_B = set(prog_B.get('completed', []))
    failed_B    = prog_B.get('failed', {})
    rows_B      = prog_B.get('rows', [])
    print(f'Track B: resuming — {len(completed_B)} done, '
          f'{len(failed_B)} failed')
else:
    completed_B = set()
    failed_B    = {}
    rows_B      = []
    print('Track B: starting fresh')

# ── MAIN EXTRACTION LOOP ──────────────────────────────────────────────────────
patient_list = df_labels['patient_id'].tolist()
n_total      = len(patient_list)

print(f'\n{"="*60}')
print('RADIOMICS EXTRACTION')
print(f'{"="*60}')
print(f'Total patients    : {n_total}')
print(f'Track A remaining : {n_total - len(completed_A)}')
print(f'Track B remaining : {n_total - len(completed_B)}')
print()

for i, patient_id in enumerate(patient_list):
    patient_dir = patient_dirs.get(patient_id)
    if patient_dir is None:
        continue

    idh_label = label_map[patient_id]
    fold      = fold_map[patient_id]

    # ── TRACK A ───────────────────────────────────────────────────────────────
    if patient_id not in completed_A:
        feats_A, err_A = extract_features_for_patient(
            patient_id, patient_dir, TRACK_A_SEQS)

        if err_A:
            failed_A[patient_id] = err_A
        else:
            row = {'patient_id': patient_id,
                   'IDH_binary': idh_label,
                   'fold':       fold}
            row.update(feats_A)
            rows_A.append(row)
            completed_A.add(patient_id)

    # ── TRACK B ───────────────────────────────────────────────────────────────
    if patient_id not in completed_B:
        feats_B, err_B = extract_features_for_patient(
            patient_id, patient_dir, TRACK_B_SEQS)

        if err_B:
            failed_B[patient_id] = err_B
        else:
            row = {'patient_id': patient_id,
                   'IDH_binary': idh_label,
                   'fold':       fold}
            row.update(feats_B)
            rows_B.append(row)
            completed_B.add(patient_id)

    # ── PROGRESS PRINT ────────────────────────────────────────────────────────
    if (i + 1) % 10 == 0 or (i + 1) == n_total:
        print(f'  [{i+1:>3}/{n_total}] '
              f'A: {len(completed_A):>3} done / {len(failed_A):>2} failed | '
              f'B: {len(completed_B):>3} done / {len(failed_B):>2} failed')

    # ── SAVE PROGRESS EVERY 10 PATIENTS ──────────────────────────────────────
    if (i + 1) % 10 == 0 or (i + 1) == n_total:
        with open(progress_path_A, 'w') as f:
            json.dump({'completed': list(completed_A),
                       'failed':    failed_A,
                       'rows':      rows_A}, f)
        with open(progress_path_B, 'w') as f:
            json.dump({'completed': list(completed_B),
                       'failed':    failed_B,
                       'rows':      rows_B}, f)

    # ── COMMIT TO KAGGLE EVERY 50 PATIENTS ───────────────────────────────────
    if (i + 1) % 50 == 0:
        print(f'  Committing progress to Kaggle...')
        commit_dir = f'{WORKING}/rad_commit'
        os.makedirs(commit_dir, exist_ok=True)

        with open(os.path.join(commit_dir,
                               'dataset-metadata.json'), 'w') as f:
            json.dump({'title':     'Glioma IDH Models',
                       'id':        MODELS_DATASET,
                       'licenses':  [{'name': 'other'}],
                       'isPrivate': True}, f)

        for fname in os.listdir(WORKING):
            src = os.path.join(WORKING, fname)
            lnk = os.path.join(commit_dir, fname)
            if (fname.startswith('cnn_track') or
                    fname.startswith('radiomics_')) and \
               (fname.endswith('.pth') or
                fname.endswith('.json') or
                fname.endswith('.csv')):
                if os.path.exists(src) and not os.path.lexists(lnk):
                    os.symlink(os.path.abspath(src), lnk)

        r = subprocess.run(
            ['kaggle', 'datasets', 'version',
             '-p', commit_dir,
             '-m', f'Radiomics progress: '
                   f'A={len(completed_A)} B={len(completed_B)}',
             '--dir-mode', 'zip'],
            capture_output=True, text=True
        )
        status = '✅' if r.returncode == 0 else '⚠️'
        print(f'  {status} Kaggle commit '
              f'(A={len(completed_A)}, B={len(completed_B)})')
        shutil.rmtree(commit_dir, ignore_errors=True)

# ── SAVE FINAL CSVs ───────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('SAVING FINAL CSVs')
print(f'{"="*60}')

df_A = pd.DataFrame(rows_A).sort_values(
    'patient_id').reset_index(drop=True)
df_B = pd.DataFrame(rows_B).sort_values(
    'patient_id').reset_index(drop=True)

csv_path_A = f'{WORKING}/radiomics_trackA.csv'
csv_path_B = f'{WORKING}/radiomics_trackB.csv'

df_A.to_csv(csv_path_A, index=False)
df_B.to_csv(csv_path_B, index=False)

print(f'Track A : {len(df_A)} patients × '
      f'{len(df_A.columns)-3} features')
print(f'Track B : {len(df_B)} patients × '
      f'{len(df_B.columns)-3} features')

if failed_A:
    print(f'\nTrack A failed ({len(failed_A)}):')
    for pid, reason in list(failed_A.items())[:5]:
        print(f'  {pid}: {reason}')
if failed_B:
    print(f'\nTrack B failed ({len(failed_B)}):')
    for pid, reason in list(failed_B.items())[:5]:
        print(f'  {pid}: {reason}')

# ── FINAL KAGGLE COMMIT ───────────────────────────────────────────────────────
print(f'\nFinal commit to Kaggle...')
commit_dir = f'{WORKING}/rad_commit_final'
os.makedirs(commit_dir, exist_ok=True)

with open(os.path.join(commit_dir, 'dataset-metadata.json'), 'w') as f:
    json.dump({'title':     'Glioma IDH Models',
               'id':        MODELS_DATASET,
               'licenses':  [{'name': 'other'}],
               'isPrivate': True}, f)

for fname in os.listdir(WORKING):
    src = os.path.join(WORKING, fname)
    lnk = os.path.join(commit_dir, fname)
    if (fname.startswith('cnn_track') or
            fname.startswith('radiomics_')) and \
       (fname.endswith('.pth') or
        fname.endswith('.json') or
        fname.endswith('.csv')):
        if os.path.exists(src) and not os.path.lexists(lnk):
            os.symlink(os.path.abspath(src), lnk)

r = subprocess.run(
    ['kaggle', 'datasets', 'version',
     '-p', commit_dir,
     '-m', f'Radiomics complete — '
           f'A:{len(df_A)} B:{len(df_B)} patients',
     '--dir-mode', 'zip'],
    capture_output=True, text=True
)

if r.returncode == 0:
    print('✅ Radiomics CSVs permanently saved to Kaggle')
else:
    print(f'⚠️  Commit failed: {r.stderr[:200]}')
    print('CSVs are in /kaggle/working/ — safe until session ends')

shutil.rmtree(commit_dir, ignore_errors=True)

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CELL 7 COMPLETE')
print(f'{"="*60}')
print(f'Track A : {len(df_A)}/495 patients, '
      f'{len(df_A.columns)-3} features')
print(f'Track B : {len(df_B)}/495 patients, '
      f'{len(df_B.columns)-3} features')
print(f'Failed  : A={len(failed_A)}, B={len(failed_B)}')
print()
print('Improvements over basic Cell 7:')
print('  ✅ Commits to Kaggle every 50 patients (resume-safe)')
print('  ✅ Restores existing progress from Kaggle on startup')
print('  ✅ Both tracks extracted in single pass')
print()
print('Next: Cell 8 — Feature selection + SVM/XGBoost + SHAP')

## Cell 8: RADIOMICS CLASSIFICATION + SHAP


In [ ]:
# ── CELL 8: RADIOMICS CLASSIFICATION + SHAP ───────────────────────────────────

# Feature selection pipeline per fold:
#   VarianceThreshold → LassoCV → 20-78 features
# Classifiers: SVM + XGBoost (both tested, best reported)
# SHAP: explains which features matter most
# Saves: radiomics_trackA_results.json
#        radiomics_trackB_results.json
#        radiomics_trackA_shap.csv
#        radiomics_trackB_shap.csv

import os, json, shutil, subprocess, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

WORKING        = '/kaggle/working'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')

# ── STEP 1: CHECK IF RESULTS ALREADY EXIST ───────────────────────────────────
print('Checking for existing Cell 8 results...')

# Download from Kaggle first
sync_dir = f'{WORKING}/cell8_sync'
os.makedirs(sync_dir, exist_ok=True)

subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

# Copy relevant files to working
for fname in os.listdir(sync_dir):
    src = os.path.join(sync_dir, fname)
    dst = os.path.join(WORKING, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

shutil.rmtree(sync_dir, ignore_errors=True)

results_A_path = f'{WORKING}/radiomics_trackA_results.json'
results_B_path = f'{WORKING}/radiomics_trackB_results.json'

# ── STEP 2: IF RESULTS EXIST — DISPLAY AND STOP ──────────────────────────────
if os.path.exists(results_A_path) and \
   os.path.exists(results_B_path):

    print('✅ Cell 8 results found — loading...\n')

    with open(results_A_path) as f:
        res_A = json.load(f)
    with open(results_B_path) as f:
        res_B = json.load(f)

    # ── DISPLAY TRACK A ───────────────────────────────────────
    print('='*60)
    print('RADIOMICS TRACK A — RESULTS')
    print('='*60)

    svm_aucs_A, xgb_aucs_A = [], []
    svm_sens_A, xgb_sens_A = [], []
    svm_spec_A, xgb_spec_A = [], []

    for fold in range(5):
        if str(fold) in res_A['fold_results']:
            r   = res_A['fold_results'][str(fold)]
            svm = r['svm_metrics']
            xgb = r['xgb_metrics']
            svm_aucs_A.append(svm['patient_auc'])
            xgb_aucs_A.append(xgb['patient_auc'])
            svm_sens_A.append(svm['sensitivity'])
            xgb_sens_A.append(xgb['sensitivity'])
            svm_spec_A.append(svm['specificity'])
            xgb_spec_A.append(xgb['specificity'])
            print(f'\nFold {fold} '
                  f'({r["n_features"]} features selected):')
            print(f'  SVM     — '
                  f'AUC: {svm["patient_auc"]:.4f} | '
                  f'Sens: {svm["sensitivity"]:.4f} | '
                  f'Spec: {svm["specificity"]:.4f} | '
                  f'F1: {svm["f1"]:.4f}')
            print(f'  XGBoost — '
                  f'AUC: {xgb["patient_auc"]:.4f} | '
                  f'Sens: {xgb["sensitivity"]:.4f} | '
                  f'Spec: {xgb["specificity"]:.4f} | '
                  f'F1: {xgb["f1"]:.4f}')
            print(f'  Best    : {r["best"]}')

    print(f'\nSVM Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in svm_aucs_A]}')
    print(f'  Mean AUC     : '
          f'{np.mean(svm_aucs_A):.4f} ± '
          f'{np.std(svm_aucs_A):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(svm_sens_A):.4f} ± '
          f'{np.std(svm_sens_A):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(svm_spec_A):.4f} ± '
          f'{np.std(svm_spec_A):.4f}')

    print(f'\nXGBoost Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in xgb_aucs_A]}')
    print(f'  Mean AUC     : '
          f'{np.mean(xgb_aucs_A):.4f} ± '
          f'{np.std(xgb_aucs_A):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(xgb_sens_A):.4f} ± '
          f'{np.std(xgb_sens_A):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(xgb_spec_A):.4f} ± '
          f'{np.std(xgb_spec_A):.4f}')

    # Feature consistency Track A
    feature_counts_A = {}
    for f in range(5):
        if str(f) in res_A['fold_results']:
            for fname in res_A['fold_results'][str(f)][
                    'feature_names']:
                feature_counts_A[fname] = \
                    feature_counts_A.get(fname, 0) + 1

    print(f'\nTop 10 most consistently selected features:')
    for fname, count in sorted(
            feature_counts_A.items(),
            key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {count}/5 folds — {fname}')

    # ── DISPLAY TRACK B ───────────────────────────────────────
    print(f'\n{"="*60}')
    print('RADIOMICS TRACK B — RESULTS')
    print('='*60)

    svm_aucs_B, xgb_aucs_B = [], []
    svm_sens_B, xgb_sens_B = [], []
    svm_spec_B, xgb_spec_B = [], []

    for fold in range(5):
        if str(fold) in res_B['fold_results']:
            r   = res_B['fold_results'][str(fold)]
            svm = r['svm_metrics']
            xgb = r['xgb_metrics']
            svm_aucs_B.append(svm['patient_auc'])
            xgb_aucs_B.append(xgb['patient_auc'])
            svm_sens_B.append(svm['sensitivity'])
            xgb_sens_B.append(xgb['sensitivity'])
            svm_spec_B.append(svm['specificity'])
            xgb_spec_B.append(xgb['specificity'])
            print(f'\nFold {fold} '
                  f'({r["n_features"]} features selected):')
            print(f'  SVM     — '
                  f'AUC: {svm["patient_auc"]:.4f} | '
                  f'Sens: {svm["sensitivity"]:.4f} | '
                  f'Spec: {svm["specificity"]:.4f} | '
                  f'F1: {svm["f1"]:.4f}')
            print(f'  XGBoost — '
                  f'AUC: {xgb["patient_auc"]:.4f} | '
                  f'Sens: {xgb["sensitivity"]:.4f} | '
                  f'Spec: {xgb["specificity"]:.4f} | '
                  f'F1: {xgb["f1"]:.4f}')
            print(f'  Best    : {r["best"]}')

    print(f'\nSVM Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in svm_aucs_B]}')
    print(f'  Mean AUC     : '
          f'{np.mean(svm_aucs_B):.4f} ± '
          f'{np.std(svm_aucs_B):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(svm_sens_B):.4f} ± '
          f'{np.std(svm_sens_B):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(svm_spec_B):.4f} ± '
          f'{np.std(svm_spec_B):.4f}')

    print(f'\nXGBoost Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in xgb_aucs_B]}')
    print(f'  Mean AUC     : '
          f'{np.mean(xgb_aucs_B):.4f} ± '
          f'{np.std(xgb_aucs_B):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(xgb_sens_B):.4f} ± '
          f'{np.std(xgb_sens_B):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(xgb_spec_B):.4f} ± '
          f'{np.std(xgb_spec_B):.4f}')

    # Feature consistency Track B
    feature_counts_B = {}
    for f in range(5):
        if str(f) in res_B['fold_results']:
            for fname in res_B['fold_results'][str(f)][
                    'feature_names']:
                feature_counts_B[fname] = \
                    feature_counts_B.get(fname, 0) + 1

    print(f'\nTop 10 most consistently selected features:')
    for fname, count in sorted(
            feature_counts_B.items(),
            key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {count}/5 folds — {fname}')

    # ── FINAL COMPARISON ──────────────────────────────────────
    best_A = max(np.mean(svm_aucs_A), np.mean(xgb_aucs_A))
    best_B = max(np.mean(svm_aucs_B), np.mean(xgb_aucs_B))
    clf_A  = ('SVM' if np.mean(svm_aucs_A) >= np.mean(xgb_aucs_A)
              else 'XGBoost')
    clf_B  = ('SVM' if np.mean(svm_aucs_B) >= np.mean(xgb_aucs_B)
              else 'XGBoost')
    std_A  = (np.std(svm_aucs_A) if clf_A == 'SVM'
              else np.std(xgb_aucs_A))
    std_B  = (np.std(svm_aucs_B) if clf_B == 'SVM'
              else np.std(xgb_aucs_B))

    print(f'\n{"="*60}')
    print('RADIOMICS vs CNN — COMPARISON')
    print(f'{"="*60}')
    print(f'\n{"Model":<30} {"AUC":>8} {"±":>4} {"STD":>8}')
    print('-'*52)
    print(f'{"CNN Track A (ResNet18)":<30} '
          f'{"0.9654":>8} {"±":>4} {"0.0190":>8}')
    print(f'{"Radiomics Track A ("+clf_A+")":<30} '
          f'{best_A:>8.4f} {"±":>4} {std_A:>8.4f}')
    print()
    print(f'{"CNN Track B (ResNet18)":<30} '
          f'{"0.9636":>8} {"±":>4} {"0.0129":>8}')
    print(f'{"Radiomics Track B ("+clf_B+")":<30} '
          f'{best_B:>8.4f} {"±":>4} {std_B:>8.4f}')
    print()
    print(f'CNN vs Radiomics delta Track A : '
          f'{0.9654 - best_A:+.4f}')
    print(f'CNN vs Radiomics delta Track B : '
          f'{0.9636 - best_B:+.4f}')
    print(f'\n✅ Cell 8 complete — results loaded from Kaggle')
    print(f'✅ No retraining needed')
    print(f'Next: Cell 9 — Hybrid Fusion embeddings')

# ── STEP 3: IF NO RESULTS — RUN FULL PIPELINE ────────────────────────────────
else:
    print('❌ No existing results found')
    print('Running full Cell 8 classification pipeline...\n')

    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.linear_model import LassoCV
    from sklearn.svm import SVC
    from sklearn.metrics import (
        roc_auc_score, accuracy_score, recall_score,
        f1_score, confusion_matrix, brier_score_loss
    )
    import xgboost as xgb
    import shap

    N_FOLDS      = 5
    CLASS_WEIGHT = 3.81
    SEED         = 42
    META_COLS    = ['patient_id', 'IDH_binary', 'fold']

    df_A = pd.read_csv(f'{WORKING}/radiomics_trackA.csv')
    df_B = pd.read_csv(f'{WORKING}/radiomics_trackB.csv')

    feature_cols_A = [c for c in df_A.columns
                      if c not in META_COLS]
    feature_cols_B = [c for c in df_B.columns
                      if c not in META_COLS]

    print(f'Track A: {len(df_A)} patients × '
          f'{len(feature_cols_A)} features')
    print(f'Track B: {len(df_B)} patients × '
          f'{len(feature_cols_B)} features')

    def compute_metrics(y_true, y_pred, y_prob):
        cm = confusion_matrix(y_true, y_pred, labels=[0,1])
        tn, fp, fn, tp = (cm.ravel() if cm.shape==(2,2)
                          else (0,0,0,0))
        return {
            'patient_auc': round(float(
                roc_auc_score(y_true, y_prob)), 4),
            'accuracy':    round(float(
                accuracy_score(y_true, y_pred)), 4),
            'sensitivity': round(float(
                recall_score(y_true, y_pred,
                             zero_division=0)), 4),
            'specificity': round(float(
                tn/(tn+fp) if (tn+fp)>0 else 0.0), 4),
            'f1':          round(float(
                f1_score(y_true, y_pred,
                         zero_division=0)), 4),
            'brier':       round(float(
                brier_score_loss(y_true, y_prob)), 4),
            'tp': int(tp), 'tn': int(tn),
            'fp': int(fp), 'fn': int(fn)
        }

    def select_features(X_train, y_train, feature_names):
        selected = list(feature_names)
        X_work   = X_train.copy()
        vt       = VarianceThreshold(threshold=0.01)
        vt.fit(X_work)
        mask_vt  = vt.get_support()
        X_work   = X_work[:, mask_vt]
        selected = [n for n, m in zip(selected, mask_vt) if m]
        print(f'    After VarianceThreshold : '
              f'{len(selected)} features')
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(X_work)
        lasso    = LassoCV(cv=3, max_iter=5000,
                           random_state=SEED, n_jobs=-1)
        lasso.fit(X_scaled, y_train)
        mask_lasso = np.abs(lasso.coef_) > 1e-6
        selected   = [n for n, m in zip(selected, mask_lasso)
                      if m]
        print(f'    After LassoCV           : '
              f'{len(selected)} features')
        return selected, vt, scaler

    def commit_to_kaggle(message):
        commit_dir = f'{WORKING}/rad8_commit'
        os.makedirs(commit_dir, exist_ok=True)
        with open(os.path.join(commit_dir,
                               'dataset-metadata.json'), 'w') as f:
            json.dump({'title':     'Glioma IDH Models',
                       'id':        MODELS_DATASET,
                       'licenses':  [{'name': 'other'}],
                       'isPrivate': True}, f)
        for fname in os.listdir(WORKING):
            src = os.path.join(WORKING, fname)
            lnk = os.path.join(commit_dir, fname)
            if any(fname.startswith(p) for p in
                   ['cnn_track', 'radiomics_']) and \
               any(fname.endswith(e) for e in
                   ['.pth', '.json', '.csv']):
                if os.path.exists(src) and \
                   not os.path.lexists(lnk):
                    os.symlink(os.path.abspath(src), lnk)
        r = subprocess.run(
            ['kaggle', 'datasets', 'version',
             '-p', commit_dir, '-m', message,
             '--dir-mode', 'zip'],
            capture_output=True, text=True)
        status = '✅' if r.returncode == 0 else '⚠️'
        print(f'  {status} Kaggle commit: {message}')
        shutil.rmtree(commit_dir, ignore_errors=True)

    def run_radiomics_cv(df, feature_cols, track_name):
        print(f'\n{"="*60}')
        print(f'RADIOMICS {track_name} — 5-FOLD CV')
        print(f'{"="*60}')

        all_val_preds   = {}
        fold_results    = {}
        shap_accumulate = []

        for fold in range(N_FOLDS):
            print(f'\nFold {fold}')
            print('-'*40)

            train_df = df[df['fold'] != fold].copy()
            val_df   = df[df['fold'] == fold].copy()

            X_train  = train_df[feature_cols].values.astype(
                np.float32)
            y_train  = train_df['IDH_binary'].values.astype(int)
            X_val    = val_df[feature_cols].values.astype(
                np.float32)
            y_val    = val_df['IDH_binary'].values.astype(int)
            pids_val = val_df['patient_id'].values

            print(f'  Train: {len(train_df)} '
                  f'({(y_train==0).sum()} WT / '
                  f'{(y_train==1).sum()} MT)')
            print(f'  Val  : {len(val_df)} '
                  f'({(y_val==0).sum()} WT / '
                  f'{(y_val==1).sum()} MT)')
            print(f'  Feature selection:')

            selected_names, vt, scaler = select_features(
                X_train, y_train, feature_cols)

            if len(selected_names) == 0:
                print(f'  WARNING: No features selected')
                continue

            mask_vt     = vt.get_support()
            X_tr_vt     = X_train[:, mask_vt]
            X_vl_vt     = X_val[:, mask_vt]
            X_tr_scaled = scaler.transform(X_tr_vt)
            X_vl_scaled = scaler.transform(X_vl_vt)

            all_after_vt  = [f for f, m in
                             zip(feature_cols, mask_vt) if m]
            lasso_indices = [i for i, n in
                             enumerate(all_after_vt)
                             if n in selected_names]

            X_tr_sel = X_tr_scaled[:, lasso_indices]
            X_vl_sel = X_vl_scaled[:, lasso_indices]
            print(f'  Final features: {X_tr_sel.shape[1]}')

            svm = SVC(kernel='rbf', probability=True,
                      class_weight={0:1.0, 1:CLASS_WEIGHT},
                      C=1.0, gamma='scale', random_state=SEED)
            svm.fit(X_tr_sel, y_train)
            svm_probs   = svm.predict_proba(X_vl_sel)[:, 1]
            svm_preds   = (svm_probs >= 0.5).astype(int)
            svm_metrics = compute_metrics(
                y_val, svm_preds, svm_probs)

            scale_pos = ((y_train==0).sum() /
                         (y_train==1).sum())
            xgb_model = xgb.XGBClassifier(
                n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos,
                eval_metric='logloss',
                random_state=SEED, n_jobs=-1)
            xgb_model.fit(X_tr_sel, y_train,
                          eval_set=[(X_vl_sel, y_val)],
                          verbose=False)
            xgb_probs   = xgb_model.predict_proba(X_vl_sel)[:, 1]
            xgb_preds   = (xgb_probs >= 0.5).astype(int)
            xgb_metrics = compute_metrics(
                y_val, xgb_preds, xgb_probs)

            print(f'  SVM     — '
                  f'AUC: {svm_metrics["patient_auc"]:.4f} | '
                  f'Sens: {svm_metrics["sensitivity"]:.4f} | '
                  f'Spec: {svm_metrics["specificity"]:.4f} | '
                  f'F1: {svm_metrics["f1"]:.4f}')
            print(f'  XGBoost — '
                  f'AUC: {xgb_metrics["patient_auc"]:.4f} | '
                  f'Sens: {xgb_metrics["sensitivity"]:.4f} | '
                  f'Spec: {xgb_metrics["specificity"]:.4f} | '
                  f'F1: {xgb_metrics["f1"]:.4f}')

            if xgb_metrics['patient_auc'] >= \
               svm_metrics['patient_auc']:
                best_probs = xgb_probs
                best_name  = 'XGBoost'
                best_model = xgb_model
            else:
                best_probs = svm_probs
                best_name  = 'SVM'
                best_model = svm

            print(f'  Best: {best_name}')

            label_map_fold = dict(zip(
                val_df['patient_id'], val_df['IDH_binary']))
            for pid, prob in zip(pids_val, best_probs):
                all_val_preds[pid] = {
                    'mean_prob': float(prob),
                    'label':     int(label_map_fold[pid]),
                    'fold':      fold
                }

            fold_results[str(fold)] = {
                'svm_metrics':   svm_metrics,
                'xgb_metrics':   xgb_metrics,
                'best':          best_name,
                'n_features':    X_tr_sel.shape[1],
                'feature_names': selected_names
            }

            try:
                explainer = shap.TreeExplainer(xgb_model)
                shap_vals = explainer.shap_values(X_vl_sel)
                for j, pid in enumerate(pids_val):
                    for k, fname in enumerate(selected_names):
                        shap_accumulate.append({
                            'patient_id':    pid,
                            'fold':          fold,
                            'feature':       fname,
                            'shap_value':    float(
                                shap_vals[j, k]),
                            'feature_value': float(
                                X_vl_sel[j, k])
                        })
            except Exception as e:
                print(f'  SHAP skipped: {e}')

        # Summary
        svm_aucs = [fold_results[str(f)]['svm_metrics'][
            'patient_auc'] for f in range(N_FOLDS)
            if str(f) in fold_results]
        xgb_aucs = [fold_results[str(f)]['xgb_metrics'][
            'patient_auc'] for f in range(N_FOLDS)
            if str(f) in fold_results]

        print(f'\n{"="*60}')
        print(f'RADIOMICS {track_name} — SUMMARY')
        print(f'{"="*60}')
        print(f'SVM     Mean AUC: '
              f'{np.mean(svm_aucs):.4f} ± '
              f'{np.std(svm_aucs):.4f}')
        print(f'XGBoost Mean AUC: '
              f'{np.mean(xgb_aucs):.4f} ± '
              f'{np.std(xgb_aucs):.4f}')

        feature_counts = {}
        for f in range(N_FOLDS):
            if str(f) in fold_results:
                for fname in fold_results[str(f)][
                        'feature_names']:
                    feature_counts[fname] = \
                        feature_counts.get(fname, 0) + 1

        print(f'\nTop 10 consistently selected features:')
        for fname, count in sorted(
                feature_counts.items(),
                key=lambda x: x[1], reverse=True)[:10]:
            print(f'  {count}/5 folds — {fname}')

        return fold_results, all_val_preds, shap_accumulate

    results_A, preds_A, shap_A = run_radiomics_cv(
        df_A, feature_cols_A, 'TRACK A')
    results_B, preds_B, shap_B = run_radiomics_cv(
        df_B, feature_cols_B, 'TRACK B')

    with open(f'{WORKING}/radiomics_trackA_results.json', 'w') as f:
        json.dump({'fold_results':  results_A,
                   'all_val_preds': preds_A}, f)
    with open(f'{WORKING}/radiomics_trackB_results.json', 'w') as f:
        json.dump({'fold_results':  results_B,
                   'all_val_preds': preds_B}, f)

    if shap_A:
        pd.DataFrame(shap_A).to_csv(
            f'{WORKING}/radiomics_trackA_shap.csv', index=False)
    if shap_B:
        pd.DataFrame(shap_B).to_csv(
            f'{WORKING}/radiomics_trackB_shap.csv', index=False)

    commit_to_kaggle(
        'Radiomics Cell 8 complete — SVM + XGBoost + SHAP')

    print(f'\n✅ Cell 8 complete')
    print(f'Next: Cell 9 — Hybrid Fusion embeddings')

## Master Summary Cell

In [5]:
# ── MASTER SUMMARY CELL — SAFE VERSION ────────────────────────────────────────
import os, json, shutil, subprocess
import numpy as np
import pandas as pd

WORKING        = '/kaggle/working'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')

print('='*60)
print('MASTER SUMMARY — RESTORING RESULTS FROM KAGGLE')
print('='*60)

# ── DOWNLOAD ALL SAVED FILES ──────────────────────────────────────────────────
sync_dir = f'{WORKING}/master_sync'
os.makedirs(sync_dir, exist_ok=True)

result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('❌ Download failed')
    print(result.stderr[:500])
    raise SystemExit

restored = []
for fname in os.listdir(sync_dir):
    src = os.path.join(sync_dir, fname)
    dst = os.path.join(WORKING, fname)
    shutil.copy(src, dst)
    restored.append(fname)

shutil.rmtree(sync_dir, ignore_errors=True)

print(f'✅ Restored {len(restored)} files')

# ── HELPERS ───────────────────────────────────────────────────────────────────
def load_json(path):
    with open(path) as f:
        return json.load(f)

def get_fold_results(obj):
    # normal case
    if isinstance(obj, dict) and 'fold_results' in obj:
        fr = obj['fold_results']
    else:
        fr = obj
    return {str(k): v for k, v in fr.items()}

def sort_fold_keys(fr):
    keys = []
    for k in fr.keys():
        try:
            keys.append((int(k), k))
        except:
            pass
    return [k for _, k in sorted(keys)]

def get_best_auc(fold_record):
    if 'best_auc' in fold_record:
        return float(fold_record['best_auc'])
    if 'best_metrics' in fold_record and 'patient_auc' in fold_record['best_metrics']:
        return float(fold_record['best_metrics']['patient_auc'])
    return np.nan

def get_metric(fold_record, metric_name):
    if 'best_metrics' in fold_record and metric_name in fold_record['best_metrics']:
        return float(fold_record['best_metrics'][metric_name])
    return np.nan

def mean_std(vals):
    vals = [v for v in vals if pd.notna(v)]
    return (np.mean(vals), np.std(vals)) if vals else (np.nan, np.nan)

# ── VERIFY FILES ──────────────────────────────────────────────────────────────
required = [
    'cnn_trackA_progress.json',
    'cnn_trackB_progress.json',
    'radiomics_trackA_results.json',
    'radiomics_trackB_results.json',
    'radiomics_trackA.csv',
    'radiomics_trackB.csv'
]
missing = [f for f in required if not os.path.exists(f'{WORKING}/{f}')]

if missing:
    print('⚠️ Missing files:', missing)
    raise SystemExit('Cannot continue without required result files')

# ── LOAD FILES ────────────────────────────────────────────────────────────────
prog_A = load_json(f'{WORKING}/cnn_trackA_progress.json')
prog_B = load_json(f'{WORKING}/cnn_trackB_progress.json')
rad_A  = load_json(f'{WORKING}/radiomics_trackA_results.json')
rad_B  = load_json(f'{WORKING}/radiomics_trackB_results.json')

df_rad_A = pd.read_csv(f'{WORKING}/radiomics_trackA.csv')
df_rad_B = pd.read_csv(f'{WORKING}/radiomics_trackB.csv')

META_COLS = ['patient_id', 'IDH_binary', 'fold']
feat_A = [c for c in df_rad_A.columns if c not in META_COLS]
feat_B = [c for c in df_rad_B.columns if c not in META_COLS]

# ── DIAGNOSTIC PRINT ──────────────────────────────────────────────────────────
print('\nCNN Track A JSON keys:', list(prog_A.keys())[:10] if isinstance(prog_A, dict) else type(prog_A))
print('CNN Track B JSON keys:', list(prog_B.keys())[:10] if isinstance(prog_B, dict) else type(prog_B))

fold_results_A = get_fold_results(prog_A)
fold_results_B = get_fold_results(prog_B)

fold_keys_A = sort_fold_keys(fold_results_A)
fold_keys_B = sort_fold_keys(fold_results_B)

print('Detected Track A folds:', fold_keys_A)
print('Detected Track B folds:', fold_keys_B)

# ── CELL 5 RESULTS — CNN TRACK A ─────────────────────────────────────────────
print(f'\n{"="*60}')
print('CELL 5 RESULTS — CNN TRACK A (7-channel ResNet18)')
print(f'{"="*60}')

aucs_A, sens_A, spec_A, f1_A, acc_A, bri_A = [], [], [], [], [], []

for k in fold_keys_A:
    r = fold_results_A[k]
    m = r.get('best_metrics', {})
    aucs_A.append(get_best_auc(r))
    sens_A.append(get_metric(r, 'sensitivity'))
    spec_A.append(get_metric(r, 'specificity'))
    f1_A.append(get_metric(r, 'f1'))
    acc_A.append(get_metric(r, 'accuracy'))
    bri_A.append(get_metric(r, 'brier'))

    print(f'\nFold {k} (best epoch {r.get("best_epoch", "NA")}):')
    print(f'  AUC         : {get_best_auc(r):.4f}')
    print(f'  Sensitivity : {m.get("sensitivity", np.nan):.4f}')
    print(f'  Specificity : {m.get("specificity", np.nan):.4f}')
    print(f'  F1          : {m.get("f1", np.nan):.4f}')
    print(f'  Accuracy    : {m.get("accuracy", np.nan):.4f}')
    print(f'  Brier       : {m.get("brier", np.nan):.4f}')
    print(f'  TP:{m.get("tp","NA")} TN:{m.get("tn","NA")} '
          f'FP:{m.get("fp","NA")} FN:{m.get("fn","NA")}')

m_auc_A, s_auc_A = mean_std(aucs_A)
m_sen_A, s_sen_A = mean_std(sens_A)
m_spe_A, s_spe_A = mean_std(spec_A)
m_f1_A,  s_f1_A  = mean_std(f1_A)
m_acc_A, s_acc_A = mean_std(acc_A)
m_bri_A, s_bri_A = mean_std(bri_A)

print(f'\nTrack A Summary:')
print(f'  Per-fold AUC : {[round(x,4) for x in aucs_A if pd.notna(x)]}')
print(f'  Mean AUC     : {m_auc_A:.4f} ± {s_auc_A:.4f}')
print(f'  Mean Sens    : {m_sen_A:.4f} ± {s_sen_A:.4f}')
print(f'  Mean Spec    : {m_spe_A:.4f} ± {s_spe_A:.4f}')
print(f'  Mean F1      : {m_f1_A:.4f} ± {s_f1_A:.4f}')
print(f'  Mean Acc     : {m_acc_A:.4f} ± {s_acc_A:.4f}')
print(f'  Mean Brier   : {m_bri_A:.4f} ± {s_bri_A:.4f}')

# ── CELL 6 RESULTS — CNN TRACK B ─────────────────────────────────────────────
print(f'\n{"="*60}')
print('CELL 6 RESULTS — CNN TRACK B (4-channel ResNet18)')
print(f'{"="*60}')

aucs_B, sens_B, spec_B, f1_B, acc_B, bri_B = [], [], [], [], [], []

for k in fold_keys_B:
    r = fold_results_B[k]
    m = r.get('best_metrics', {})
    aucs_B.append(get_best_auc(r))
    sens_B.append(get_metric(r, 'sensitivity'))
    spec_B.append(get_metric(r, 'specificity'))
    f1_B.append(get_metric(r, 'f1'))
    acc_B.append(get_metric(r, 'accuracy'))
    bri_B.append(get_metric(r, 'brier'))

    print(f'\nFold {k} (best epoch {r.get("best_epoch", "NA")}):')
    print(f'  AUC         : {get_best_auc(r):.4f}')
    print(f'  Sensitivity : {m.get("sensitivity", np.nan):.4f}')
    print(f'  Specificity : {m.get("specificity", np.nan):.4f}')
    print(f'  F1          : {m.get("f1", np.nan):.4f}')
    print(f'  Accuracy    : {m.get("accuracy", np.nan):.4f}')
    print(f'  Brier       : {m.get("brier", np.nan):.4f}')
    print(f'  TP:{m.get("tp","NA")} TN:{m.get("tn","NA")} '
          f'FP:{m.get("fp","NA")} FN:{m.get("fn","NA")}')

m_auc_B, s_auc_B = mean_std(aucs_B)
m_sen_B, s_sen_B = mean_std(sens_B)
m_spe_B, s_spe_B = mean_std(spec_B)
m_f1_B,  s_f1_B  = mean_std(f1_B)
m_acc_B, s_acc_B = mean_std(acc_B)
m_bri_B, s_bri_B = mean_std(bri_B)

print(f'\nTrack B Summary:')
print(f'  Per-fold AUC : {[round(x,4) for x in aucs_B if pd.notna(x)]}')
print(f'  Mean AUC     : {m_auc_B:.4f} ± {s_auc_B:.4f}')
print(f'  Mean Sens    : {m_sen_B:.4f} ± {s_sen_B:.4f}')
print(f'  Mean Spec    : {m_spe_B:.4f} ± {s_spe_B:.4f}')
print(f'  Mean F1      : {m_f1_B:.4f} ± {s_f1_B:.4f}')
print(f'  Mean Acc     : {m_acc_B:.4f} ± {s_acc_B:.4f}')
print(f'  Mean Brier   : {m_bri_B:.4f} ± {s_bri_B:.4f}')

# ── CELL 7 RESULTS — RADIOMICS EXTRACTION ────────────────────────────────────
print(f'\n{"="*60}')
print('CELL 7 RESULTS — RADIOMICS EXTRACTION')
print(f'{"="*60}')
print(f'Track A : {len(df_rad_A)} patients × {len(feat_A)} features')
print(f'Track B : {len(df_rad_B)} patients × {len(feat_B)} features')
print(f'Wildtype: {(df_rad_A["IDH_binary"] == 0).sum()}')
print(f'Mutant  : {(df_rad_A["IDH_binary"] == 1).sum()}')

# ── CELL 8 RESULTS — RADIOMICS CLASSIFICATION ────────────────────────────────
print(f'\n{"="*60}')
print('CELL 8 RESULTS — RADIOMICS CLASSIFICATION')
print(f'{"="*60}')

rad_fold_A = get_fold_results(rad_A['fold_results'] if 'fold_results' in rad_A else rad_A)
rad_fold_B = get_fold_results(rad_B['fold_results'] if 'fold_results' in rad_B else rad_B)

rad_keys_A = sort_fold_keys(rad_fold_A)
rad_keys_B = sort_fold_keys(rad_fold_B)

print('\nTRACK A')
rad_A_svm, rad_A_xgb = [], []
for k in rad_keys_A:
    r = rad_fold_A[k]
    svm = r['svm_metrics']
    xgb = r['xgb_metrics']
    rad_A_svm.append(float(svm['patient_auc']))
    rad_A_xgb.append(float(xgb['patient_auc']))
    print(f'\nFold {k} ({r["n_features"]} selected features):')
    print(f'  SVM     : AUC {svm["patient_auc"]:.4f} | '
          f'Sens {svm["sensitivity"]:.4f} | '
          f'Spec {svm["specificity"]:.4f} | '
          f'F1 {svm["f1"]:.4f}')
    print(f'  XGBoost : AUC {xgb["patient_auc"]:.4f} | '
          f'Sens {xgb["sensitivity"]:.4f} | '
          f'Spec {xgb["specificity"]:.4f} | '
          f'F1 {xgb["f1"]:.4f}')
    print(f'  Best    : {r["best"]}')

print(f'\nTrack A Radiomics Summary:')
print(f'  Mean SVM AUC : {np.mean(rad_A_svm):.4f} ± {np.std(rad_A_svm):.4f}')
print(f'  Mean XGB AUC : {np.mean(rad_A_xgb):.4f} ± {np.std(rad_A_xgb):.4f}')

print('\nTRACK B')
rad_B_svm, rad_B_xgb = [], []
for k in rad_keys_B:
    r = rad_fold_B[k]
    svm = r['svm_metrics']
    xgb = r['xgb_metrics']
    rad_B_svm.append(float(svm['patient_auc']))
    rad_B_xgb.append(float(xgb['patient_auc']))
    print(f'\nFold {k} ({r["n_features"]} selected features):')
    print(f'  SVM     : AUC {svm["patient_auc"]:.4f} | '
          f'Sens {svm["sensitivity"]:.4f} | '
          f'Spec {svm["specificity"]:.4f} | '
          f'F1 {svm["f1"]:.4f}')
    print(f'  XGBoost : AUC {xgb["patient_auc"]:.4f} | '
          f'Sens {xgb["sensitivity"]:.4f} | '
          f'Spec {xgb["specificity"]:.4f} | '
          f'F1 {xgb["f1"]:.4f}')
    print(f'  Best    : {r["best"]}')

print(f'\nTrack B Radiomics Summary:')
print(f'  Mean SVM AUC : {np.mean(rad_B_svm):.4f} ± {np.std(rad_B_svm):.4f}')
print(f'  Mean XGB AUC : {np.mean(rad_B_xgb):.4f} ± {np.std(rad_B_xgb):.4f}')

# ── GRAND COMPARISON ──────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('GRAND COMPARISON TABLE')
print(f'{"="*60}')
print(f'\n{"Model":<35} {"Track A":>10} {"Track B":>10}')
print('─'*57)
print(f'{"CNN ResNet18":<35} {m_auc_A:>10.4f} {m_auc_B:>10.4f}')
print(f'{"Radiomics SVM":<35} {np.mean(rad_A_svm):>10.4f} {np.mean(rad_B_svm):>10.4f}')
print(f'{"Radiomics XGBoost":<35} {np.mean(rad_A_xgb):>10.4f} {np.mean(rad_B_xgb):>10.4f}')

print(f'\nCNN vs Radiomics delta Track A : '
      f'{m_auc_A - max(np.mean(rad_A_svm), np.mean(rad_A_xgb)):+.4f}')
print(f'CNN vs Radiomics delta Track B : '
      f'{m_auc_B - max(np.mean(rad_B_svm), np.mean(rad_B_xgb)):+.4f}')

print(f'\n✅ All previous results restored and displayed')
print(f'✅ Ready to proceed to Cell 9')

MASTER SUMMARY — RESTORING RESULTS FROM KAGGLE
✅ Restored 24 files

CNN Track A JSON keys: ['completed_folds', 'fold_results', 'all_val_preds']
CNN Track B JSON keys: ['completed_folds', 'fold_results', 'all_val_preds']
Detected Track A folds: ['0', '1', '2', '3', '4']
Detected Track B folds: ['0', '1', '2', '3', '4']

CELL 5 RESULTS — CNN TRACK A (7-channel ResNet18)

Fold 0 (best epoch 18):
  AUC         : 0.9709
  Sensitivity : 0.6500
  Specificity : 0.9620
  F1          : 0.7222
  Accuracy    : 0.8990
  Brier       : 0.0640
  TP:13 TN:76 FP:3 FN:7

Fold 1 (best epoch 8):
  AUC         : 0.9411
  Sensitivity : 0.8000
  Specificity : 0.9620
  F1          : 0.8205
  Accuracy    : 0.9293
  Brier       : 0.0698
  TP:16 TN:76 FP:3 FN:4

Fold 2 (best epoch 6):
  AUC         : 0.9554
  Sensitivity : 0.8571
  Specificity : 0.8974
  F1          : 0.7660
  Accuracy    : 0.8889
  Brier       : 0.0820
  TP:18 TN:70 FP:8 FN:3

Fold 3 (best epoch 11):
  AUC         : 0.9982
  Sensitivity : 0.6667

## CELL 9: HYBRID FUSION — EMBEDDING EXTRACTION + FUSION

In [13]:
# ── CELL 9: HYBRID FUSION — EMBEDDING EXTRACTION + FUSION ────────────────────
# Early fusion: CNN embeddings + radiomics features → SVM/XGBoost
# Late fusion:  weighted average CNN prob + radiomics prob
#
# FIXES APPLIED:
# - Track letter detection fixed (exact match, not substring)
# - num_workers=0, pin_memory=False (prevents hang)
# - Softer error handling for restore step

import os, json, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
import xgboost as xgb
from kaggle_secrets import UserSecretsClient

warnings.filterwarnings('ignore')

WORKING        = '/kaggle/working'
SLICES_DIR     = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

# ── KAGGLE API SETUP ──────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
print(f'Kaggle API ready')

# ── RESTORE EVERYTHING FROM KAGGLE ───────────────────────────────────────────
print('\n' + '='*60)
print('STEP 1: RESTORING ALL SAVED FILES FROM KAGGLE')
print('='*60)

sync_dir = f'{WORKING}/cell9_sync'
os.makedirs(sync_dir, exist_ok=True)

result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

if result.returncode == 0:
    restored = []
    for fname in os.listdir(sync_dir):
        src = os.path.join(sync_dir, fname)
        dst = os.path.join(WORKING, fname)
        shutil.copy(src, dst)
        restored.append(fname)

    print(f'Restored {len(restored)} files')
else:
    print(f'⚠️ Download had issues: {result.stderr[:300]}')
    print('Continuing anyway — files may already be present...')

shutil.rmtree(sync_dir, ignore_errors=True)

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FOLDS      = 5
CLASS_WEIGHT = 3.81
SEED         = 42
BATCH_SIZE   = 64
PCA_DIMS     = 50
META_COLS    = ['patient_id', 'IDH_binary', 'fold']

CNN_WEIGHT = 0.6
RAD_WEIGHT = 0.4

print(f'\nDevice         : {DEVICE}')
print(f'PCA dims       : {PCA_DIMS}')
print(f'Late fusion    : CNN×{CNN_WEIGHT} + Radiomics×{RAD_WEIGHT}')

# ── LOAD ALL DATA ─────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 2: LOADING DATA')
print('='*60)

df_labels = pd.read_csv(f'{WORKING}/labels.csv')
df_A      = pd.read_csv(f'{WORKING}/radiomics_trackA.csv')
df_B      = pd.read_csv(f'{WORKING}/radiomics_trackB.csv')

with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

with open(f'{WORKING}/radiomics_trackA_results.json') as f:
    rad_results_A = json.load(f)
with open(f'{WORKING}/radiomics_trackB_results.json') as f:
    rad_results_B = json.load(f)

with open(f'{WORKING}/cnn_trackA_progress.json') as f:
    cnn_progress_A = json.load(f)
with open(f'{WORKING}/cnn_trackB_progress.json') as f:
    cnn_progress_B = json.load(f)

label_map = dict(zip(df_labels['patient_id'],
                     df_labels['IDH_binary'].astype(int)))

feature_cols_A = [c for c in df_A.columns if c not in META_COLS]
feature_cols_B = [c for c in df_B.columns if c not in META_COLS]

print(f'Patients       : {len(df_labels)}')
print(f'Track A feats  : {len(feature_cols_A)} radiomic features')
print(f'Track B feats  : {len(feature_cols_B)} radiomic features')
print(f'CNN Track A OOF: {len(cnn_progress_A.get("all_val_preds", {}))} patients')
print(f'CNN Track B OOF: {len(cnn_progress_B.get("all_val_preds", {}))} patients')

# ── MODEL — EMBEDDING EXTRACTOR ───────────────────────────────────────────────
def build_embedding_model(n_channels, weights_path):
    model = resnet18(weights=None)
    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    model.conv1 = new_conv
    model.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)
    )

    state = torch.load(weights_path, map_location=DEVICE)

    # Safety check
    ckpt_channels = state['conv1.weight'].shape[1]
    if ckpt_channels != n_channels:
        raise RuntimeError(
            f'Channel mismatch: checkpoint has {ckpt_channels} channels, '
            f'model expects {n_channels}. Check weights_path: {weights_path}'
        )

    model.load_state_dict(state)
    model.fc = nn.Identity()
    model = model.to(DEVICE)
    model.eval()
    return model

# ── DATASET ───────────────────────────────────────────────────────────────────
class EmbeddingDataset(torch.utils.data.Dataset):
    def __init__(self, patient_ids, slices_dir, n_channels):
        self.slices_dir = slices_dir
        self.n_channels = n_channels
        self.items      = []
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            arr = np.load(npy_path, mmap_mode='r')
            for s in range(arr.shape[0]):
                self.items.append((pid, s))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s = self.items[idx]
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        return x, pid

# ── EXTRACT EMBEDDINGS ────────────────────────────────────────────────────────
def extract_embeddings(patient_ids, model, slices_dir,
                       n_channels, device):
    dataset = EmbeddingDataset(patient_ids, slices_dir, n_channels)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0,
                         pin_memory=False)

    patient_embeddings = {}

    with torch.no_grad():
        for x, pids in loader:
            x    = x.to(device)
            embs = model(x).cpu().numpy()
            for emb, pid in zip(embs, pids):
                patient_embeddings.setdefault(pid, []).append(emb)

    return {
        pid: np.mean(np.stack(embs), axis=0)
        for pid, embs in patient_embeddings.items()
    }

# ── METRICS ───────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = (cm.ravel() if cm.shape == (2, 2)
                      else (0, 0, 0, 0))
    return {
        'patient_auc': round(float(
            roc_auc_score(y_true, y_prob)), 4),
        'accuracy':    round(float(
            accuracy_score(y_true, y_pred)), 4),
        'sensitivity': round(float(
            recall_score(y_true, y_pred,
                         zero_division=0)), 4),
        'specificity': round(float(
            tn / (tn + fp) if (tn + fp) > 0 else 0.0), 4),
        'f1':          round(float(
            f1_score(y_true, y_pred,
                     zero_division=0)), 4),
        'brier':       round(float(
            brier_score_loss(y_true, y_prob)), 4),
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    }

# ── FEATURE SELECTION ─────────────────────────────────────────────────────────
def select_features(X_train, y_train, feature_names):
    selected = list(feature_names)
    X_work   = X_train.copy()

    vt = VarianceThreshold(threshold=0.01)
    vt.fit(X_work)
    mask_vt  = vt.get_support()
    X_work   = X_work[:, mask_vt]
    selected = [n for n, m in zip(selected, mask_vt) if m]

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_work)

    lasso = LassoCV(cv=3, max_iter=5000,
                    random_state=SEED, n_jobs=-1)
    lasso.fit(X_scaled, y_train)
    mask_lasso = np.abs(lasso.coef_) > 1e-6
    selected   = [n for n, m in zip(selected, mask_lasso) if m]

    return selected, vt, scaler

# ── KAGGLE COMMIT ─────────────────────────────────────────────────────────────
def commit_to_kaggle(message):
    commit_dir = f'{WORKING}/fusion_commit'
    os.makedirs(commit_dir, exist_ok=True)

    with open(os.path.join(commit_dir,
                           'dataset-metadata.json'), 'w') as f:
        json.dump({'title':     'Glioma IDH Models',
                   'id':        MODELS_DATASET,
                   'licenses':  [{'name': 'other'}],
                   'isPrivate': True}, f)

    for fname in os.listdir(WORKING):
        src = os.path.join(WORKING, fname)
        lnk = os.path.join(commit_dir, fname)
        if any(fname.startswith(p) for p in
               ['cnn_track', 'radiomics_', 'fusion_']) and \
           any(fname.endswith(e) for e in
               ['.pth', '.json', '.csv']):
            if os.path.exists(src) and not os.path.lexists(lnk):
                os.symlink(os.path.abspath(src), lnk)

    for fname in ['labels.csv', 'patient_splits.json']:
        src = os.path.join(WORKING, fname)
        lnk = os.path.join(commit_dir, fname)
        if os.path.exists(src) and not os.path.lexists(lnk):
            os.symlink(os.path.abspath(src), lnk)

    r = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', commit_dir, '-m', message,
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
    status = '✅' if r.returncode == 0 else '⚠️'
    print(f'  {status} Kaggle: {message}')
    shutil.rmtree(commit_dir, ignore_errors=True)

# ── MAIN FUSION FUNCTION ──────────────────────────────────────────────────────
def run_fusion(track_name, n_channels, df_rad,
               feature_cols, cnn_oof, rad_oof):
    print(f'\n{"="*60}')
    print(f'HYBRID FUSION — {track_name}')
    print(f'{"="*60}')

    # FIX: exact match, not substring
    if track_name == 'TRACK A':
        track_letter = 'A'
    else:
        track_letter = 'B'

    fold_results  = {}
    all_val_preds = {}

    for fold in range(N_FOLDS):
        print(f'\nFold {fold}')
        print('-'*40)

        val_patients   = splits[f'fold_{fold}']
        train_patients = [p for f2 in range(N_FOLDS)
                          if f2 != fold
                          for p in splits[f'fold_{f2}']]

        # FIX: correct path using track_letter
        weights_path = f'{WORKING}/cnn_track{track_letter}_fold{fold}_best.pth'

        if not os.path.exists(weights_path):
            print(f'  ❌ Weights missing: {weights_path}')
            continue

        model = build_embedding_model(n_channels, weights_path)
        print(f'  Weights loaded ✅')
        print(f'  Extracting train embeddings '
              f'({len(train_patients)} patients)...')

        train_embs = extract_embeddings(
            train_patients, model, SLICES_DIR,
            n_channels, DEVICE)

        print(f'  Extracting val embeddings '
              f'({len(val_patients)} patients)...')

        val_embs = extract_embeddings(
            val_patients, model, SLICES_DIR,
            n_channels, DEVICE)

        del model
        torch.cuda.empty_cache()

        train_pids = [p for p in train_patients
                      if p in train_embs and
                      p in df_rad['patient_id'].values]
        val_pids   = [p for p in val_patients
                      if p in val_embs and
                      p in df_rad['patient_id'].values]

        X_emb_train = np.stack(
            [train_embs[p] for p in train_pids])
        X_emb_val   = np.stack(
            [val_embs[p] for p in val_pids])

        y_train = np.array([label_map[p] for p in train_pids])
        y_val   = np.array([label_map[p] for p in val_pids])

        n_components = min(PCA_DIMS,
                           X_emb_train.shape[1],
                           X_emb_train.shape[0] - 1)
        pca = PCA(n_components=n_components, random_state=SEED)
        X_pca_train = pca.fit_transform(X_emb_train)
        X_pca_val   = pca.transform(X_emb_val)
        var_explained = pca.explained_variance_ratio_.sum() * 100

        print(f'  PCA: 512 → {n_components} dims '
              f'({var_explained:.1f}% variance explained)')

        rad_train = df_rad[
            df_rad['patient_id'].isin(train_pids)
        ].set_index('patient_id')
        rad_val = df_rad[
            df_rad['patient_id'].isin(val_pids)
        ].set_index('patient_id')

        X_rad_train = rad_train.loc[
            train_pids, feature_cols].values.astype(np.float32)
        X_rad_val   = rad_val.loc[
            val_pids,   feature_cols].values.astype(np.float32)

        selected, vt, scaler = select_features(
            X_rad_train, y_train, feature_cols)

        mask_vt      = vt.get_support()
        all_after_vt = [f for f, m in zip(feature_cols, mask_vt)
                        if m]
        lasso_idx    = [i for i, n in enumerate(all_after_vt)
                        if n in selected]

        X_rad_tr_vt  = X_rad_train[:, mask_vt]
        X_rad_vl_vt  = X_rad_val[:,   mask_vt]
        X_rad_tr_sc  = scaler.transform(X_rad_tr_vt)
        X_rad_vl_sc  = scaler.transform(X_rad_vl_vt)
        X_rad_tr_sel = X_rad_tr_sc[:, lasso_idx]
        X_rad_vl_sel = X_rad_vl_sc[:, lasso_idx]

        print(f'  Radiomics: {len(selected)} features selected')

        X_fused_train = np.concatenate(
            [X_pca_train, X_rad_tr_sel], axis=1)
        X_fused_val   = np.concatenate(
            [X_pca_val,   X_rad_vl_sel], axis=1)

        print(f'  Fused dims: {X_fused_train.shape[1]} '
              f'({n_components} PCA + '
              f'{len(selected)} radiomics)')

        svm = SVC(kernel='rbf', probability=True,
                  class_weight={0: 1.0, 1: CLASS_WEIGHT},
                  C=1.0, gamma='scale', random_state=SEED)
        svm.fit(X_fused_train, y_train)
        svm_probs = svm.predict_proba(X_fused_val)[:, 1]
        svm_preds = (svm_probs >= 0.5).astype(int)
        svm_m     = compute_metrics(y_val, svm_preds, svm_probs)

        scale_pos = ((y_train == 0).sum() /
                     max((y_train == 1).sum(), 1))
        xgb_model = xgb.XGBClassifier(
            n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos,
            eval_metric='logloss',
            random_state=SEED, n_jobs=-1
        )
        xgb_model.fit(
            X_fused_train, y_train,
            eval_set=[(X_fused_val, y_val)],
            verbose=False
        )
        xgb_probs = xgb_model.predict_proba(X_fused_val)[:, 1]
        xgb_preds = (xgb_probs >= 0.5).astype(int)
        xgb_m     = compute_metrics(y_val, xgb_preds, xgb_probs)

        late_equal_probs    = []
        late_weighted_probs = []
        late_labels         = []

        for pid in val_pids:
            if pid in cnn_oof and pid in rad_oof:
                cnn_p = cnn_oof[pid]['mean_prob']
                rad_p = rad_oof[pid]['mean_prob']
                late_equal_probs.append(
                    0.5 * cnn_p + 0.5 * rad_p)
                late_weighted_probs.append(
                    CNN_WEIGHT * cnn_p + RAD_WEIGHT * rad_p)
                late_labels.append(label_map[pid])

        late_equal_preds    = [1 if p >= 0.5 else 0
                               for p in late_equal_probs]
        late_weighted_preds = [1 if p >= 0.5 else 0
                               for p in late_weighted_probs]

        late_equal_m = (compute_metrics(
            late_labels, late_equal_preds, late_equal_probs)
            if late_labels else {})
        late_weighted_m = (compute_metrics(
            late_labels, late_weighted_preds, late_weighted_probs)
            if late_labels else {})

        print(f'\n  Results:')
        print(f'  Early Fusion SVM     : '
              f'AUC {svm_m["patient_auc"]:.4f} | '
              f'Sens {svm_m["sensitivity"]:.4f} | '
              f'Spec {svm_m["specificity"]:.4f} | '
              f'F1 {svm_m["f1"]:.4f}')
        print(f'  Early Fusion XGBoost : '
              f'AUC {xgb_m["patient_auc"]:.4f} | '
              f'Sens {xgb_m["sensitivity"]:.4f} | '
              f'Spec {xgb_m["specificity"]:.4f} | '
              f'F1 {xgb_m["f1"]:.4f}')
        if late_equal_m:
            print(f'  Late Fusion (equal)  : '
                  f'AUC {late_equal_m["patient_auc"]:.4f} | '
                  f'Sens {late_equal_m["sensitivity"]:.4f} | '
                  f'Spec {late_equal_m["specificity"]:.4f} | '
                  f'F1 {late_equal_m["f1"]:.4f}')
        if late_weighted_m:
            print(f'  Late Fusion (wtd)    : '
                  f'AUC {late_weighted_m["patient_auc"]:.4f} | '
                  f'Sens {late_weighted_m["sensitivity"]:.4f} | '
                  f'Spec {late_weighted_m["specificity"]:.4f} | '
                  f'F1 {late_weighted_m["f1"]:.4f}')

        if xgb_m['patient_auc'] >= svm_m['patient_auc']:
            best_early_probs = xgb_probs
            best_early_name  = 'XGBoost'
        else:
            best_early_probs = svm_probs
            best_early_name  = 'SVM'

        print(f'  Best early fusion    : {best_early_name}')

        for pid, prob in zip(val_pids, best_early_probs):
            all_val_preds[pid] = {
                'mean_prob':  float(prob),
                'label':      int(label_map[pid]),
                'fold':       fold
            }

        fold_results[str(fold)] = {
            'early_svm':          svm_m,
            'early_xgb':          xgb_m,
            'late_equal':         late_equal_m,
            'late_weighted':      late_weighted_m,
            'best_early':         best_early_name,
            'n_pca_dims':         int(n_components),
            'n_rad_features':     len(selected),
            'n_fused_dims':       int(X_fused_train.shape[1]),
            'pca_var_explained':  round(float(var_explained), 2)
        }

        _, used, free = shutil.disk_usage(WORKING)
        print(f'  Disk: {used/1e9:.2f} GB used / '
              f'{free/1e9:.2f} GB free')

    # Summary
    print(f'\n{"="*60}')
    print(f'FUSION {track_name} — SUMMARY')
    print(f'{"="*60}')

    def mean_std(metric, sub_key):
        vals = [fold_results[str(f)][metric][sub_key]
                for f in range(N_FOLDS)
                if str(f) in fold_results and
                fold_results[str(f)][metric]]
        return np.mean(vals), np.std(vals)

    svm_auc,      svm_std      = mean_std('early_svm',     'patient_auc')
    xgb_auc,      xgb_std      = mean_std('early_xgb',     'patient_auc')
    late_eq_auc,  late_eq_std  = mean_std('late_equal',    'patient_auc')
    late_wtd_auc, late_wtd_std = mean_std('late_weighted', 'patient_auc')

    print(f'\n{"Method":<28} {"AUC":>8}  {"±STD":>8}')
    print('─'*48)
    print(f'{"Early Fusion SVM":<28} '
          f'{svm_auc:>8.4f}  {svm_std:>8.4f}')
    print(f'{"Early Fusion XGBoost":<28} '
          f'{xgb_auc:>8.4f}  {xgb_std:>8.4f}')
    print(f'{"Late Fusion (equal)":<28} '
          f'{late_eq_auc:>8.4f}  {late_eq_std:>8.4f}')
    print(f'{"Late Fusion (weighted)":<28} '
          f'{late_wtd_auc:>8.4f}  {late_wtd_std:>8.4f}')

    return fold_results, all_val_preds

# ── RUN FUSION — TRACK A ──────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 3: RUNNING FUSION')
print('='*60)

results_fusion_A, preds_fusion_A = run_fusion(
    track_name   = 'TRACK A',
    n_channels   = 7,
    df_rad       = df_A,
    feature_cols = feature_cols_A,
    cnn_oof      = cnn_progress_A.get('all_val_preds', {}),
    rad_oof      = rad_results_A.get('all_val_preds', {})
)

# ── RUN FUSION — TRACK B ──────────────────────────────────────────────────────
results_fusion_B, preds_fusion_B = run_fusion(
    track_name   = 'TRACK B',
    n_channels   = 4,
    df_rad       = df_B,
    feature_cols = feature_cols_B,
    cnn_oof      = cnn_progress_B.get('all_val_preds', {}),
    rad_oof      = rad_results_B.get('all_val_preds', {})
)

# ── SAVE RESULTS ──────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('STEP 4: SAVING RESULTS')
print('='*60)

with open(f'{WORKING}/fusion_trackA_results.json', 'w') as f:
    json.dump({'fold_results':  results_fusion_A,
               'all_val_preds': preds_fusion_A}, f)

with open(f'{WORKING}/fusion_trackB_results.json', 'w') as f:
    json.dump({'fold_results':  results_fusion_B,
               'all_val_preds': preds_fusion_B}, f)

print('Saved: fusion_trackA_results.json')
print('Saved: fusion_trackB_results.json')

commit_to_kaggle('Cell 9 complete — Hybrid Fusion Track A + B')

# ── GRAND SUMMARY — ALL ARMS ──────────────────────────────────────────────────
print(f'\n{"="*60}')
print('GRAND SUMMARY — ALL ARMS')
print(f'{"="*60}')

def get_mean_auc(results, metric_key):
    vals = [results[str(f)][metric_key]['patient_auc']
            for f in range(5)
            if str(f) in results and results[str(f)][metric_key]]
    return np.mean(vals), np.std(vals)

cnn_A_aucs = [cnn_progress_A['fold_results'][str(f)]['best_auc']
              for f in range(5)]
cnn_B_aucs = [cnn_progress_B['fold_results'][str(f)]['best_auc']
              for f in range(5)]

rad_A_svm  = [rad_results_A['fold_results'][str(f)]['svm_metrics']['patient_auc']
              for f in range(5)]
rad_A_xgb  = [rad_results_A['fold_results'][str(f)]['xgb_metrics']['patient_auc']
              for f in range(5)]
rad_B_svm  = [rad_results_B['fold_results'][str(f)]['svm_metrics']['patient_auc']
              for f in range(5)]
rad_B_xgb  = [rad_results_B['fold_results'][str(f)]['xgb_metrics']['patient_auc']
              for f in range(5)]

fus_A_svm,  fus_A_svm_s  = get_mean_auc(results_fusion_A, 'early_svm')
fus_A_xgb,  fus_A_xgb_s  = get_mean_auc(results_fusion_A, 'early_xgb')
fus_A_late, fus_A_late_s = get_mean_auc(results_fusion_A, 'late_weighted')
fus_B_svm,  fus_B_svm_s  = get_mean_auc(results_fusion_B, 'early_svm')
fus_B_xgb,  fus_B_xgb_s  = get_mean_auc(results_fusion_B, 'early_xgb')
fus_B_late, fus_B_late_s = get_mean_auc(results_fusion_B, 'late_weighted')

print(f'\n{"Model":<32} {"Track A AUC":>12} {"Track B AUC":>12}')
print('─'*58)
print(f'{"CNN (ResNet18)":<32} '
      f'{np.mean(cnn_A_aucs):>10.4f}   '
      f'{np.mean(cnn_B_aucs):>10.4f}')
print(f'{"Radiomics SVM":<32} '
      f'{np.mean(rad_A_svm):>10.4f}   '
      f'{np.mean(rad_B_svm):>10.4f}')
print(f'{"Radiomics XGBoost":<32} '
      f'{np.mean(rad_A_xgb):>10.4f}   '
      f'{np.mean(rad_B_xgb):>10.4f}')
print(f'{"Early Fusion SVM":<32} '
      f'{fus_A_svm:>10.4f}   '
      f'{fus_B_svm:>10.4f}')
print(f'{"Early Fusion XGBoost":<32} '
      f'{fus_A_xgb:>10.4f}   '
      f'{fus_B_xgb:>10.4f}')
print(f'{"Late Fusion (weighted)":<32} '
      f'{fus_A_late:>10.4f}   '
      f'{fus_B_late:>10.4f}')
print()
print(f'Literature benchmark          : ~0.90')
print()
print(f'✅ Cell 9 complete')
print(f'✅ Results saved to Kaggle permanently')
print(f'Next: Cell 10 — External Validation (EGD + TCGA)')

Kaggle API ready

STEP 1: RESTORING ALL SAVED FILES FROM KAGGLE
Restored 22 files

Device         : cuda
PCA dims       : 50
Late fusion    : CNN×0.6 + Radiomics×0.4

STEP 2: LOADING DATA
Patients       : 495
Track A feats  : 616 radiomic features
Track B feats  : 352 radiomic features
CNN Track A OOF: 495 patients
CNN Track B OOF: 495 patients

STEP 3: RUNNING FUSION

HYBRID FUSION — TRACK A

Fold 0
----------------------------------------
  Weights loaded ✅
  Extracting train embeddings (396 patients)...
  Extracting val embeddings (99 patients)...
  PCA: 512 → 50 dims (99.8% variance explained)
  Radiomics: 53 features selected
  Fused dims: 103 (50 PCA + 53 radiomics)

  Results:
  Early Fusion SVM     : AUC 0.9772 | Sens 0.8500 | Spec 0.9620 | F1 0.8500
  Early Fusion XGBoost : AUC 0.9297 | Sens 0.6500 | Spec 0.9620 | F1 0.7222
  Late Fusion (equal)  : AUC 0.9829 | Sens 0.7500 | Spec 0.9873 | F1 0.8333
  Late Fusion (wtd)    : AUC 0.9810 | Sens 0.8000 | Spec 0.9620 | F1 0.8205
  B

In [8]:
import pandas as pd
import os

# ── EGD LABELS ────────────────────────────────────────────
print('='*60)
print('EGD LABELS')
print('='*60)

egd_label_path = '/kaggle/input/datasets/adesaladaniel/egd-labels/Genetic_and_Histological_labels.xlsx'
df_egd = pd.read_excel(egd_label_path)

print(f'Shape: {df_egd.shape}')
print(f'Columns: {list(df_egd.columns)}')
print(f'\nFirst 5 rows:')
print(df_egd.head())
print(f'\nIDH value counts:')
print(df_egd.iloc[:, -1].value_counts())

# ── BRATS LABELS ──────────────────────────────────────────
print('\n' + '='*60)
print('BRATS 2021 LABELS')
print('='*60)

brats_label_path = '/kaggle/input/datasets/adesaladaniel/tcgabrats2021-idh-labels/TCGA_BraTS2021_IDH_labels.csv'
df_brats = pd.read_csv(brats_label_path)

print(f'Shape: {df_brats.shape}')
print(f'Columns: {list(df_brats.columns)}')
print(f'\nFirst 5 rows:')
print(df_brats.head())
print(f'\nIDH value counts:')
print(df_brats['IDH_binary'].value_counts() 
      if 'IDH_binary' in df_brats.columns 
      else df_brats.iloc[:, -1].value_counts())

# ── EGD FOLDER STRUCTURE ──────────────────────────────────
print('\n' + '='*60)
print('EGD PATIENT FOLDER STRUCTURE')
print('='*60)

egd_batch1 = '/kaggle/input/datasets/adesaladaniel/egd-batch-01'
first_patient = sorted(os.listdir(egd_batch1))[0]
patient_path  = os.path.join(egd_batch1, first_patient)

print(f'First patient: {first_patient}')
print(f'Contents:')
for item in sorted(os.listdir(patient_path)):
    full = os.path.join(patient_path, item)
    if os.path.isdir(full):
        sub = os.listdir(full)
        print(f'  📂 {item}/ ({len(sub)} files)')
        for s in sorted(sub)[:5]:
            print(f'       — {s}')
    else:
        size = round(os.path.getsize(full)/1e6, 2)
        print(f'  📄 {item} ({size} MB)')

# ── BRATS TAR CHECK ───────────────────────────────────────
print('\n' + '='*60)
print('BRATS TAR FILE CHECK')
print('='*60)

brats_dir = '/kaggle/input/datasets/dschettler8845/brats-2021-task1'
for f in sorted(os.listdir(brats_dir)):
    full = os.path.join(brats_dir, f)
    size = round(os.path.getsize(full)/1e9, 2)
    print(f'  {f}  —  {size} GB')

EGD LABELS
Shape: (774, 4)
Columns: ['Subject', 'IDH', '1p19q', 'Grade']

First 5 rows:
    Subject  IDH  1p19q  Grade
0  EGD-0001   -1     -1      4
1  EGD-0002   -1     -1      2
2  EGD-0003   -1     -1      4
3  EGD-0004    1      1      2
4  EGD-0005   -1     -1      4

IDH value counts:
Grade
 4    502
 2    135
 3     79
-1     58
Name: count, dtype: int64

BRATS 2021 LABELS
Shape: (140, 5)
Columns: ['BraTS_ID', 'TCGA_barcode', 'collection', 'IDH_binary', 'complete_label']

First 5 rows:
          BraTS_ID  TCGA_barcode collection  IDH_binary      complete_label
0  BraTS2021_00100  TCGA-02-0085   TCGA-GBM         0.0  glioblastoma_IDHwt
1  BraTS2021_00102  TCGA-02-0102   TCGA-GBM         0.0  glioblastoma_IDHwt
2  BraTS2021_00106  TCGA-19-2631   TCGA-GBM         0.0  glioblastoma_IDHwt
3  BraTS2021_00107  TCGA-76-6280   TCGA-GBM         0.0  glioblastoma_IDHwt
4  BraTS2021_00108  TCGA-76-6193   TCGA-GBM         0.0  glioblastoma_IDHwt

IDH value counts:
IDH_binary
0.0    84
1.0  

In [9]:
import pandas as pd
import os

egd_label_path = '/kaggle/input/datasets/adesaladaniel/egd-labels/Genetic_and_Histological_labels.xlsx'
df_egd = pd.read_excel(egd_label_path)

print('IDH column value counts:')
print(df_egd['IDH'].value_counts().sort_index())
print(f'\nTotal patients     : {len(df_egd)}')
print(f'IDH known (0 or 1) : {(df_egd["IDH"].isin([0,1])).sum()}')
print(f'IDH wildtype (0)   : {(df_egd["IDH"] == 0).sum()}')
print(f'IDH mutant (1)     : {(df_egd["IDH"] == 1).sum()}')
print(f'IDH unknown (-1)   : {(df_egd["IDH"] == -1).sum()}')

print(f'\nGrade distribution (known IDH only):')
df_known = df_egd[df_egd['IDH'].isin([0, 1])]
print(df_known['Grade'].value_counts().sort_index())

# Check folders
egd_batches = [
    f'/kaggle/input/datasets/adesaladaniel/egd-batch-0{i}'
    for i in range(1, 10)
] + ['/kaggle/input/datasets/adesaladaniel/egd-batch-10']

patient_dirs = {}
for batch in egd_batches:
    if os.path.exists(batch):
        for entry in sorted(os.listdir(batch)):
            if entry.startswith('EGD-'):
                patient_dirs[entry] = os.path.join(batch, entry)

print(f'\nEGD folders found  : {len(patient_dirs)}')

# Match
matched = df_egd[
    df_egd['Subject'].isin(patient_dirs.keys()) &
    df_egd['IDH'].isin([0, 1])
]
print(f'Matched with labels: {len(matched)}')
print(f'  Wildtype : {(matched["IDH"] == 0).sum()}')
print(f'  Mutant   : {(matched["IDH"] == 1).sum()}')

IDH column value counts:
IDH
-1    307
 0    312
 1    155
Name: count, dtype: int64

Total patients     : 774
IDH known (0 or 1) : 467
IDH wildtype (0)   : 312
IDH mutant (1)     : 155
IDH unknown (-1)   : 307

Grade distribution (known IDH only):
Grade
-1     52
 2    127
 3     29
 4    259
Name: count, dtype: int64

EGD folders found  : 467
Matched with labels: 467
  Wildtype : 312
  Mutant   : 155


***VALIDATION PREPROCESSING***

In [10]:
# ── EGD: PREPROCESS TRACK B TENSORS + SAVE TO KAGGLE ─────────────────────────
# Track B only: T1, T1GD(T1c), T2, FLAIR
# Output per patient: (n_slices, 4, 224, 224) float16
# Saves permanently to: adesaladaniel/egd-idh-tensors

import os, json, shutil, subprocess
import numpy as np
import pandas as pd
import nibabel as nib
from skimage.transform import resize
from kaggle_secrets import UserSecretsClient

WORKING = '/kaggle/working'
EGD_OUT_DIR = f'{WORKING}/egd_tensors'
os.makedirs(EGD_OUT_DIR, exist_ok=True)

EGD_DATASET_ID = 'adesaladaniel/egd-idh-tensors'
EGD_TITLE      = 'EGD IDH Tensors'

SLICE_SIZE = 224
TUMOR_THR  = 0.001

# ── KAGGLE AUTH ───────────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
print('Kaggle API ready')

# ── HELPERS ───────────────────────────────────────────────────────────────────
def load_volume(path):
    return nib.load(path).get_fdata(dtype=np.float32)

def clip_and_normalize(vol, brain_mask):
    vox = vol[brain_mask > 0]
    if vox.size == 0:
        return vol
    p1, p99 = np.percentile(vox, 1), np.percentile(vox, 99)
    vol = np.clip(vol, p1, p99)
    vox = vol[brain_mask > 0]
    mean, std = vox.mean(), vox.std()
    if std < 1e-8:
        out = np.zeros_like(vol)
    else:
        out = (vol - mean) / std
    out[brain_mask == 0] = 0.0
    return out

def resize_slice(slc):
    return resize(
        slc, (SLICE_SIZE, SLICE_SIZE),
        order=1, mode='constant',
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

def select_tumor_slices(mask):
    n_slices = mask.shape[2]
    area = mask.shape[0] * mask.shape[1]
    return [
        s for s in range(n_slices)
        if (mask[:, :, s] > 0).sum() / area >= TUMOR_THR
    ]

def upload_or_version_dataset(dataset_dir, dataset_id, title, message):
    meta_path = os.path.join(dataset_dir, 'dataset-metadata.json')
    with open(meta_path, 'w') as f:
        json.dump({
            'title': title,
            'id': dataset_id,
            'licenses': [{'name': 'other'}],
            'isPrivate': True
        }, f)

    create_cmd = ['kaggle', 'datasets', 'create', '-p', dataset_dir, '--dir-mode', 'tar']
    r = subprocess.run(create_cmd, capture_output=True, text=True)

    if r.returncode == 0:
        print(f'✅ Created dataset: {dataset_id}')
        return True

    # If dataset exists already, version it
    version_cmd = ['kaggle', 'datasets', 'version', '-p', dataset_dir, '-m', message, '--dir-mode', 'tar']
    r2 = subprocess.run(version_cmd, capture_output=True, text=True)

    if r2.returncode == 0:
        print(f'✅ Updated dataset: {dataset_id}')
        return True

    print('⚠️ Upload/version failed')
    print(r2.stderr[:500] if r2.stderr else r.stderr[:500])
    return False

# ── LOAD LABELS ───────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 1: LOAD EGD LABELS + FOLDERS')
print('='*60)

label_path = '/kaggle/input/datasets/adesaladaniel/egd-labels/Genetic_and_Histological_labels.xlsx'
df = pd.read_excel(label_path)

# keep only known IDH
df = df[df['IDH'].isin([0, 1])].copy()
df.rename(columns={'Subject': 'patient_id', 'IDH': 'IDH_binary'}, inplace=True)

egd_batches = [
    f'/kaggle/input/datasets/adesaladaniel/egd-batch-0{i}'
    for i in range(1, 10)
] + ['/kaggle/input/datasets/adesaladaniel/egd-batch-10']

patient_dirs = {}
for batch in egd_batches:
    if not os.path.exists(batch):
        continue
    for entry in sorted(os.listdir(batch)):
        if entry.startswith('EGD-'):
            patient_dirs[entry] = os.path.join(batch, entry)

df = df[df['patient_id'].isin(patient_dirs)].reset_index(drop=True)
df['folder_path'] = df['patient_id'].map(patient_dirs)

print(f'Patients with labels + folders: {len(df)}')
print(f'Wildtype: {(df["IDH_binary"] == 0).sum()}')
print(f'Mutant  : {(df["IDH_binary"] == 1).sum()}')

# ── RESUME SUPPORT ────────────────────────────────────────────────────────────
progress_path = f'{WORKING}/egd_tensor_progress.json'
if os.path.exists(progress_path):
    with open(progress_path) as f:
        prog = json.load(f)
    completed = set(prog.get('completed', []))
    failed    = prog.get('failed', {})
    print(f'Resuming — completed: {len(completed)}, failed: {len(failed)}')
else:
    completed = set()
    failed    = {}
    print('Starting fresh')

# ── PREPROCESS ────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 2: PREPROCESS EGD PATIENTS')
print('='*60)

remaining = [pid for pid in df['patient_id'].tolist()
             if pid not in completed and pid not in failed]

for i, pid in enumerate(remaining):
    folder = df.loc[df['patient_id'] == pid, 'folder_path'].values[0]
    try:
        # EGD filenames
        t1_path    = os.path.join(folder, 'T1.nii')
        t1c_path   = os.path.join(folder, 'T1GD.nii')   # T1GD = T1c
        t2_path    = os.path.join(folder, 'T2.nii')
        flair_path = os.path.join(folder, 'FLAIR.nii')
        mask_path  = os.path.join(folder, 'MASK.nii')

        for p in [t1_path, t1c_path, t2_path, flair_path, mask_path]:
            if not os.path.exists(p):
                raise FileNotFoundError(os.path.basename(p))

        tumor_mask = (load_volume(mask_path) > 0).astype(np.float32)
        selected = select_tumor_slices(tumor_mask)
        if len(selected) == 0:
            raise ValueError('No tumor slices above threshold')

        channel_arrays = []
        for seq_path in [t1_path, t1c_path, t2_path, flair_path]:
            vol = load_volume(seq_path)
            brain_mask = (vol != 0).astype(np.float32)
            vol = clip_and_normalize(vol, brain_mask)
            seq_slices = np.stack(
                [resize_slice(vol[:, :, s]) for s in selected],
                axis=0
            )
            channel_arrays.append(seq_slices)

        tensor = np.stack(channel_arrays, axis=1).astype(np.float16)
        np.save(os.path.join(EGD_OUT_DIR, f'{pid}.npy'), tensor)

        completed.add(pid)
        print(f'[{i+1}/{len(remaining)}] OK   {pid} — {tensor.shape[0]} slices')

    except Exception as e:
        failed[pid] = str(e)
        print(f'[{i+1}/{len(remaining)}] FAIL {pid} — {e}')

    if (i + 1) % 10 == 0 or (i + 1) == len(remaining):
        with open(progress_path, 'w') as f:
            json.dump({
                'completed': list(completed),
                'failed': failed
            }, f)

# ── SAVE LABEL CSV ────────────────────────────────────────────────────────────
df_saved = df[df['patient_id'].isin(completed)].copy()
df_saved[['patient_id', 'IDH_binary', '1p19q', 'Grade']].to_csv(
    os.path.join(EGD_OUT_DIR, 'egd_labels.csv'),
    index=False
)

print('\nPreprocessing summary:')
print(f'Completed: {len(completed)}')
print(f'Failed   : {len(failed)}')

# ── UPLOAD ────────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 3: UPLOAD EGD TENSORS TO KAGGLE')
print('='*60)

ok = upload_or_version_dataset(
    EGD_OUT_DIR,
    EGD_DATASET_ID,
    EGD_TITLE,
    f'EGD tensors update — {len(completed)} patients'
)

# ── FINAL ─────────────────────────────────────────────────────────────────────
npy_files = [f for f in os.listdir(EGD_OUT_DIR) if f.endswith('.npy')]
print('\n' + '='*60)
print('EGD TENSOR PREP COMPLETE')
print('='*60)
print(f'Tensors saved : {len(npy_files)}')
print(f'Dataset       : {EGD_DATASET_ID}')
print(f'Upload status : {"SUCCESS" if ok else "FAILED"}')
print('\nNext: run the BraTS tensor prep cell')

Kaggle API ready

STEP 1: LOAD EGD LABELS + FOLDERS
Patients with labels + folders: 467
Wildtype: 312
Mutant  : 155
Starting fresh

STEP 2: PREPROCESS EGD PATIENTS
[1/467] OK   EGD-0004 — 48 slices
[2/467] OK   EGD-0008 — 67 slices
[3/467] OK   EGD-0009 — 31 slices
[4/467] OK   EGD-0011 — 66 slices
[5/467] OK   EGD-0014 — 61 slices
[6/467] OK   EGD-0015 — 101 slices
[7/467] FAIL EGD-0020 — Expected 34701156 bytes, got 9932203 bytes from /kaggle/input/datasets/adesaladaniel/egd-batch-01/EGD-0020/T1.nii
 - could the file be damaged?
[8/467] OK   EGD-0022 — 41 slices
[9/467] OK   EGD-0024 — 93 slices
[10/467] OK   EGD-0026 — 93 slices
[11/467] OK   EGD-0029 — 44 slices
[12/467] OK   EGD-0031 — 80 slices
[13/467] OK   EGD-0033 — 67 slices
[14/467] OK   EGD-0034 — 104 slices
[15/467] OK   EGD-0035 — 75 slices
[16/467] OK   EGD-0041 — 108 slices
[17/467] OK   EGD-0045 — 89 slices
[18/467] OK   EGD-0047 — 30 slices
[19/467] OK   EGD-0050 — 93 slices
[20/467] OK   EGD-0052 — 101 slices
[21/467

In [11]:
# ── BRATS 2021: SELECTIVE EXTRACT + TRACK B TENSORS + SAVE TO KAGGLE ─────────
# Extracts only labeled BraTS patients from TAR
# Track B only: t1, t1ce(T1c), t2, flair
# Output per patient: (n_slices, 4, 224, 224) float16
# Saves permanently to: adesaladaniel/brats2021-idh-tensors

import os, re, json, tarfile, shutil, subprocess
import numpy as np
import pandas as pd
import nibabel as nib
from skimage.transform import resize
from kaggle_secrets import UserSecretsClient

WORKING = '/kaggle/working'
BRATS_RAW_DIR = f'{WORKING}/brats2021_labeled_raw'
BRATS_OUT_DIR = f'{WORKING}/brats2021_tensors'
os.makedirs(BRATS_RAW_DIR, exist_ok=True)
os.makedirs(BRATS_OUT_DIR, exist_ok=True)

BRATS_DATASET_ID = 'adesaladaniel/brats2021-idh-tensors'
BRATS_TITLE      = 'BraTS2021 IDH Tensors'

SLICE_SIZE = 224
TUMOR_THR  = 0.001

# ── KAGGLE AUTH ───────────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
print('Kaggle API ready')

# ── PATHS ─────────────────────────────────────────────────────────────────────
LABEL_PATH = '/kaggle/input/datasets/adesaladaniel/tcgabrats2021-idh-labels/TCGA_BraTS2021_IDH_labels.csv'
TAR_PATH   = '/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar'

# ── HELPERS ───────────────────────────────────────────────────────────────────
def load_volume(path):
    return nib.load(path).get_fdata(dtype=np.float32)

def clip_and_normalize(vol, brain_mask):
    vox = vol[brain_mask > 0]
    if vox.size == 0:
        return vol
    p1, p99 = np.percentile(vox, 1), np.percentile(vox, 99)
    vol = np.clip(vol, p1, p99)
    vox = vol[brain_mask > 0]
    mean, std = vox.mean(), vox.std()
    if std < 1e-8:
        out = np.zeros_like(vol)
    else:
        out = (vol - mean) / std
    out[brain_mask == 0] = 0.0
    return out

def resize_slice(slc):
    return resize(
        slc, (SLICE_SIZE, SLICE_SIZE),
        order=1, mode='constant',
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

def select_tumor_slices(mask):
    n_slices = mask.shape[2]
    area = mask.shape[0] * mask.shape[1]
    return [
        s for s in range(n_slices)
        if (mask[:, :, s] > 0).sum() / area >= TUMOR_THR
    ]

def upload_or_version_dataset(dataset_dir, dataset_id, title, message):
    meta_path = os.path.join(dataset_dir, 'dataset-metadata.json')
    with open(meta_path, 'w') as f:
        json.dump({
            'title': title,
            'id': dataset_id,
            'licenses': [{'name': 'other'}],
            'isPrivate': True
        }, f)

    create_cmd = ['kaggle', 'datasets', 'create', '-p', dataset_dir, '--dir-mode', 'tar']
    r = subprocess.run(create_cmd, capture_output=True, text=True)

    if r.returncode == 0:
        print(f'✅ Created dataset: {dataset_id}')
        return True

    version_cmd = ['kaggle', 'datasets', 'version', '-p', dataset_dir, '-m', message, '--dir-mode', 'tar']
    r2 = subprocess.run(version_cmd, capture_output=True, text=True)

    if r2.returncode == 0:
        print(f'✅ Updated dataset: {dataset_id}')
        return True

    print('⚠️ Upload/version failed')
    print(r2.stderr[:500] if r2.stderr else r.stderr[:500])
    return False

# ── LOAD LABELS ───────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 1: LOAD BRATS LABELS')
print('='*60)

df = pd.read_csv(LABEL_PATH)
df = df[df['IDH_binary'].isin([0, 1, 0.0, 1.0])].copy()
df['IDH_binary'] = df['IDH_binary'].astype(int)
df['BraTS_ID']   = df['BraTS_ID'].astype(str)
target_ids = set(df['BraTS_ID'].tolist())

print(f'Labeled patients: {len(df)}')
print(f'Wildtype        : {(df["IDH_binary"] == 0).sum()}')
print(f'Mutant          : {(df["IDH_binary"] == 1).sum()}')

# ── SELECTIVE EXTRACTION ──────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 2: SELECTIVE TAR EXTRACTION')
print('='*60)

# If already extracted, skip
already_dirs = {}
for root, dirs, files in os.walk(BRATS_RAW_DIR):
    for d in dirs:
        if re.fullmatch(r'BraTS2021_\d{5}', d):
            already_dirs[d] = os.path.join(root, d)

if len(already_dirs) >= max(1, int(0.9 * len(target_ids))):
    print(f'Already extracted: {len(already_dirs)} patient folders')
else:
    pattern = re.compile(r'BraTS2021_\d{5}')
    members_to_extract = []
    matched_ids = set()

    print('Scanning TAR...')
    with tarfile.open(TAR_PATH, 'r') as tar:
        for member in tar.getmembers():
            match = pattern.search(member.name)
            if match:
                pid = match.group(0)
                if pid in target_ids:
                    members_to_extract.append(member)
                    matched_ids.add(pid)

        print(f'Matched labeled IDs: {len(matched_ids)}')
        print(f'Members to extract : {len(members_to_extract)}')
        print('Extracting...')
        tar.extractall(path=BRATS_RAW_DIR, members=members_to_extract)

# discover extracted dirs
patient_dirs = {}
for root, dirs, files in os.walk(BRATS_RAW_DIR):
    for d in dirs:
        if re.fullmatch(r'BraTS2021_\d{5}', d):
            patient_dirs[d] = os.path.join(root, d)

print(f'Extracted patient folders: {len(patient_dirs)}')

# ── RESUME SUPPORT ────────────────────────────────────────────────────────────
progress_path = f'{WORKING}/brats_tensor_progress.json'
if os.path.exists(progress_path):
    with open(progress_path) as f:
        prog = json.load(f)
    completed = set(prog.get('completed', []))
    failed    = prog.get('failed', {})
    print(f'Resuming — completed: {len(completed)}, failed: {len(failed)}')
else:
    completed = set()
    failed    = {}
    print('Starting fresh preprocessing')

# ── PREPROCESS ────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 3: PREPROCESS BRATS PATIENTS')
print('='*60)

def find_file(patient_dir, suffix):
    for f in os.listdir(patient_dir):
        if f.endswith(f'_{suffix}.nii.gz') or f.endswith(f'_{suffix}.nii'):
            return os.path.join(patient_dir, f)
    return None

remaining = [pid for pid in sorted(patient_dirs.keys())
             if pid not in completed and pid not in failed]

for i, pid in enumerate(remaining):
    folder = patient_dirs[pid]
    try:
        t1_path    = find_file(folder, 't1')
        t1c_path   = find_file(folder, 't1ce')   # t1ce = T1c
        t2_path    = find_file(folder, 't2')
        flair_path = find_file(folder, 'flair')
        seg_path   = find_file(folder, 'seg')

        for p in [t1_path, t1c_path, t2_path, flair_path, seg_path]:
            if p is None or not os.path.exists(p):
                raise FileNotFoundError(str(p))

        tumor_mask = (load_volume(seg_path) > 0).astype(np.float32)
        selected = select_tumor_slices(tumor_mask)
        if len(selected) == 0:
            raise ValueError('No tumor slices above threshold')

        channel_arrays = []
        for seq_path in [t1_path, t1c_path, t2_path, flair_path]:
            vol = load_volume(seq_path)
            brain_mask = (vol != 0).astype(np.float32)
            vol = clip_and_normalize(vol, brain_mask)
            seq_slices = np.stack(
                [resize_slice(vol[:, :, s]) for s in selected],
                axis=0
            )
            channel_arrays.append(seq_slices)

        tensor = np.stack(channel_arrays, axis=1).astype(np.float16)
        np.save(os.path.join(BRATS_OUT_DIR, f'{pid}.npy'), tensor)

        completed.add(pid)
        print(f'[{i+1}/{len(remaining)}] OK   {pid} — {tensor.shape[0]} slices')

    except Exception as e:
        failed[pid] = str(e)
        print(f'[{i+1}/{len(remaining)}] FAIL {pid} — {e}')

    if (i + 1) % 10 == 0 or (i + 1) == len(remaining):
        with open(progress_path, 'w') as f:
            json.dump({
                'completed': list(completed),
                'failed': failed
            }, f)

# ── SAVE LABEL CSV ────────────────────────────────────────────────────────────
df_saved = df[df['BraTS_ID'].isin(completed)].copy()
df_saved.to_csv(os.path.join(BRATS_OUT_DIR, 'brats_labels.csv'), index=False)

print('\nPreprocessing summary:')
print(f'Completed: {len(completed)}')
print(f'Failed   : {len(failed)}')

# ── UPLOAD ────────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STEP 4: UPLOAD BRATS TENSORS TO KAGGLE')
print('='*60)

ok = upload_or_version_dataset(
    BRATS_OUT_DIR,
    BRATS_DATASET_ID,
    BRATS_TITLE,
    f'BraTS tensors update — {len(completed)} patients'
)

# ── OPTIONAL CLEANUP ──────────────────────────────────────────────────────────
print('\nDisk usage after upload attempt:')
total, used, free = shutil.disk_usage(WORKING)
print(f'  Used : {used/1e9:.2f} GB')
print(f'  Free : {free/1e9:.2f} GB')

# ── FINAL ─────────────────────────────────────────────────────────────────────
npy_files = [f for f in os.listdir(BRATS_OUT_DIR) if f.endswith('.npy')]
print('\n' + '='*60)
print('BRATS TENSOR PREP COMPLETE')
print('='*60)
print(f'Tensors saved : {len(npy_files)}')
print(f'Dataset       : {BRATS_DATASET_ID}')
print(f'Upload status : {"SUCCESS" if ok else "FAILED"}')

print('\nIMPORTANT:')
print('- This saves CNN-ready tensors only.')
print('- For full BraTS radiomics/fusion external validation later,')
print('  we will either need raw extracted files in this same session')
print('  OR precompute BraTS radiomics features before ending the session.')

Kaggle API ready

STEP 1: LOAD BRATS LABELS
Labeled patients: 140
Wildtype        : 84
Mutant          : 56

STEP 2: SELECTIVE TAR EXTRACTION
Scanning TAR...
Matched labeled IDs: 140
Members to extract : 840
Extracting...


/tmp/ipykernel_57/2464588031.py:146: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=BRATS_RAW_DIR, members=members_to_extract)


Extracted patient folders: 140
Starting fresh preprocessing

STEP 3: PREPROCESS BRATS PATIENTS
[1/140] OK   BraTS2021_00100 — 66 slices
[2/140] OK   BraTS2021_00102 — 27 slices
[3/140] OK   BraTS2021_00106 — 73 slices
[4/140] OK   BraTS2021_00107 — 52 slices
[5/140] OK   BraTS2021_00108 — 56 slices
[6/140] OK   BraTS2021_00109 — 68 slices
[7/140] OK   BraTS2021_00111 — 71 slices
[8/140] OK   BraTS2021_00113 — 42 slices
[9/140] OK   BraTS2021_00116 — 90 slices
[10/140] OK   BraTS2021_00117 — 66 slices
[11/140] OK   BraTS2021_00120 — 58 slices
[12/140] OK   BraTS2021_00121 — 79 slices
[13/140] OK   BraTS2021_00122 — 46 slices
[14/140] OK   BraTS2021_00123 — 87 slices
[15/140] OK   BraTS2021_00128 — 66 slices
[16/140] OK   BraTS2021_00130 — 78 slices
[17/140] OK   BraTS2021_00132 — 66 slices
[18/140] OK   BraTS2021_00133 — 68 slices
[19/140] OK   BraTS2021_00134 — 73 slices
[20/140] OK   BraTS2021_00136 — 46 slices
[21/140] OK   BraTS2021_00137 — 70 slices
[22/140] OK   BraTS2021_00138 — 